# Integrated Multi-Source Feature Analysis — Multi-Region

Builds an **integrated feature table** per study area (Cordilleran / Superior /
Grenville) by assigning every data source to each point, then runs PCA + LDA.

**Data sources → how each is assigned to a point**

| Source | Geometry | Method |
|---|---|---|
| REE occurrences | points | positive samples (label = 1) |
| Geochemistry | points | negatives = far samples; values via IDW (positives) / measured (negatives) |
| Terrane | polygons | point-in-polygon → `geo_age`, `appearance`, `crustal_type` |
| Structure | lines | distance to nearest line → `struct_proximity` (0–1) |
| Gravity / Magnetic | raster | sample pixel value at each point (**RASTER hook**) |

**Per region it produces:** integrated feature CSV, PCA figure (scatter +
density + biplot), LDA figure + cross-validated AUC, and a cross-region summary.


## 1. Configuration

In [ ]:
# PROJ 数据库路径修复 — 必须在所有 import 之前运行
import os, importlib.util
_spec = importlib.util.find_spec('rasterio')
if _spec:
    _proj_data = os.path.join(os.path.dirname(_spec.origin), 'proj_data')
    os.environ['PROJ_DATA'] = _proj_data
    os.environ['PROJ_LIB']  = _proj_data
    print(f"PROJ_DATA -> {_proj_data}")

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy.stats import gaussian_kde
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_val_score

BASE = '../dataset'

# Shared rasters (national scale, same file for every region)
_RASTERS = {
    # ── Getech Gravity (.flt) ──────────────────────────────────────────
    'gravity_ba':          f'{BASE}/converted getech data/Gravity_converted/Canada_BA.flt',
    'gravity_ba_1vd':      f'{BASE}/converted getech data/Gravity_converted/Canada_BA_1VD.flt',
    'gravity_ba_hp100km':  f'{BASE}/converted getech data/Gravity_converted/Canada_BA_HP100km.flt',
    'gravity_ba_hp40km':   f'{BASE}/converted getech data/Gravity_converted/Canada_BA_HP40km.flt',
    'gravity_ba_thd':      f'{BASE}/converted getech data/Gravity_converted/Canada_BA_THD.flt',
    'gravity_ba_thd_tilt': f'{BASE}/converted getech data/Gravity_converted/Canada_BA_THD_Tilt.flt',
    # ── Getech Magnetic (.flt) ─────────────────────────────────────────
    'mag_drtp':            f'{BASE}/converted getech data/Magnetic_converted/DRTP.flt',
    'mag_drtp_1vd':        f'{BASE}/converted getech data/Magnetic_converted/DRTP_1VD.flt',
    'mag_drtp_thd':        f'{BASE}/converted getech data/Magnetic_converted/DRTP_THD.flt',
    'mag_drtp_thd_tilt':   f'{BASE}/converted getech data/Magnetic_converted/DRTP_THD_Tilt.flt',
    'mag_tmi':             f'{BASE}/converted getech data/Magnetic_converted/TMI.flt',
    # ── USGS deep-crustal rasters (.tif) ──────────────────────────────
    'gravity_up30km':      f'{BASE}/new USGS data/GeophysicsGravity_Up30km_USCanada.tif',
    'gravity_up30km_hgm':  f'{BASE}/new USGS data/GeophysicsGravity_Up30km_HGM_USCanada.tif',
    'lab_depth':           f'{BASE}/new USGS data/GeophysicsLAB_USCanada.tif',
    'mag_rtp_deep':        f'{BASE}/new USGS data/USCanadaMagRTP_DeepSources.tif',
    'mag_rtp_hgm_deep':    f'{BASE}/new USGS data/USCanadaMagRTP_HGMDeepSources.tif',
    'moho_depth':          f'{BASE}/new USGS data/USCanada_Moho.tif',
}

REGIONS = {
    'Cordilleran': {
        'geo':     None,   # no geochemistry data
        'dep':     f'{BASE}/Carbonatite_REE_Canada/REE_Canada_Carbonatite.csv',
        'terrane': f'{BASE}/new USGS data/Geology_H3Grid_Canada.shp',
        'struct':  f'{BASE}/Faults_Canada/Faults_Canada.geojson',
        'outline': f'{BASE}/3_areas/Cordilleran.geojson',
        'raster':  _RASTERS,
    },
    'Superior': {
        'geo':     None,
        'dep':     f'{BASE}/Carbonatite_REE_Canada/REE_Canada_Carbonatite.csv',
        'terrane': f'{BASE}/new USGS data/Geology_H3Grid_Canada.shp',
        'struct':  f'{BASE}/Faults_Canada/Faults_Canada.geojson',
        'outline': f'{BASE}/3_areas/Superior.geojson',
        'raster':  _RASTERS,
    },
    'Grenville': {
        'geo':     None,
        'dep':     f'{BASE}/Carbonatite_REE_Canada/REE_Canada_Carbonatite.csv',
        'terrane': f'{BASE}/new USGS data/Geology_H3Grid_Canada.shp',
        'struct':  f'{BASE}/Faults_Canada/Faults_Canada.geojson',
        'outline': f'{BASE}/3_areas/Grenville.geojson',
        'raster':  _RASTERS,
    },
}

# ── Parameters ────────────────────────────────────────────────────────────
PROJECTED_CRS = 'EPSG:3978'   # Canada Atlas Lambert (metres)
BUFFER_KM     = 10.0          # min distance from any REE deposit to be a negative sample
N_BACKGROUND  = 400           # max negative samples per region
KNN_IMPUTE_K  = 5
RANDOM_SEED   = 42


## 1.5. Data Inventory — Raster & Vector Layers

In [ ]:
print('=' * 60)
print('Missing value analysis — raw geochemistry')
print('=' * 60)

for region_name, cfg in REGIONS.items():
    geo_path = cfg.get('geo')
    if not geo_path:
        print(f'\n[{region_name}] geo path not set — skipping')
        continue

    # First pass: read only the header to find which ELEMENT_COLS actually exist
    header = pd.read_csv(geo_path, nrows=0)
    elems = [c for c in ELEMENT_COLS if c in header.columns]
    coord_cols = [c for c in ['latitude', 'longitude'] if c in header.columns]
    if not elems:
        print(f'\n[{region_name}] none of ELEMENT_COLS found in CSV — skipping')
        continue

    # Second pass: read only the columns we need
    geo = pd.read_csv(geo_path, usecols=coord_cols + elems)

    print(f"\n{'─'*60}")
    print(f"Region: {region_name}  |  Total samples: {len(geo)}  |  Elements found: {len(elems)}/{len(ELEMENT_COLS)}")
    print(f"{'─'*60}")

    missing = geo[elems].isna().sum()
    pct = missing / len(geo) * 100
    summary = pd.DataFrame({
        'Missing (n)': missing,
        'Missing (%)': pct.round(1),
        'Present (n)': len(geo) - missing,
    }).sort_values('Missing (%)', ascending=False)
    print(summary.to_string())

    del geo  # free memory immediately after use

    # Bar chart
    fig, ax = plt.subplots(figsize=(12, 4))
    colors = ['#c0392b' if p > 50 else '#e67e22' if p > 20 else '#27ae60'
              for p in pct[summary.index]]
    ax.bar([ELEM_LABEL.get(c, c) for c in summary.index], summary['Missing (%)'], color=colors)
    ax.set_ylabel('Missing (%)')
    ax.set_title(f'{region_name}: Missing value rate per element (sorted descending)')
    ax.axhline(20, color='orange', lw=1, ls='--', label='20% threshold')
    ax.axhline(50, color='red',    lw=1, ls='--', label='50% threshold')
    ax.legend()
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


## 1.6. Preprocess — Clip National Data to Study Area Boundaries

In [ ]:
import rasterio
from rasterio.mask import mask as rio_mask

CLIP_DIR = 'clipped'  # clipped files are cached here; delete to force re-clip

def clip_raster_to_boundary(src_path, boundary_gdf, out_path):
    """Clip a raster to a boundary polygon and save as GeoTIFF."""
    import pyproj
    with rasterio.open(src_path) as src:
        if src.crs:
            bnd = boundary_gdf.to_crs(src.crs)
        else:
            bnd = boundary_gdf  # assume raster is already EPSG:4326
            print(f'      WARNING: {src_path} has no CRS — assuming EPSG:4326')
        shapes = [geom.__geo_interface__ for geom in bnd.geometry]
        out_img, out_transform = rio_mask(src, shapes, crop=True)
        meta = src.meta.copy()
        meta.update(driver='GTiff',
                    height=out_img.shape[1],
                    width=out_img.shape[2],
                    transform=out_transform)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with rasterio.open(out_path, 'w', **meta) as dst:
        dst.write(out_img)


def preprocess_region(name, cfg):
    """Clip all rasters and vector layers for one region to its outline."""
    outline_path = cfg.get('outline')
    if not outline_path:
        return cfg
    boundary = gpd.read_file(outline_path).to_crs('EPSG:4326')
    out_dir = os.path.join(CLIP_DIR, name)
    os.makedirs(out_dir, exist_ok=True)
    new_cfg = dict(cfg)  # shallow copy

    # ── Rasters ──────────────────────────────────────────────────────────
    new_raster = {}
    for feat, path in cfg.get('raster', {}).items():
        out_path = os.path.join(out_dir, f'{feat}.tif')
        if not os.path.exists(out_path):
            print(f'  clipping raster  {feat} ...')
            clip_raster_to_boundary(path, boundary, out_path)
        else:
            print(f'  cached raster    {feat}')
        new_raster[feat] = out_path
    new_cfg['raster'] = new_raster

    # ── Vector layers (terrane, struct) ───────────────────────────────────
    for key in ('terrane', 'struct'):
        vec_path = cfg.get(key)
        if not vec_path:
            continue
        out_path = os.path.join(out_dir, f'{key}.geojson')
        if not os.path.exists(out_path):
            print(f'  clipping vector  {key} ...')
            gdf = gpd.read_file(vec_path).to_crs('EPSG:4326')
            gpd.clip(gdf, boundary).to_file(out_path, driver='GeoJSON')
        else:
            print(f'  cached vector    {key}')
        new_cfg[key] = out_path

    return new_cfg


# ── Run preprocessing for all configured regions ──────────────────────────
print(f'Clipping national data → {CLIP_DIR}/')
print('(Already-clipped files are reused; delete the folder to force re-clip)\n')
REGIONS_CLIPPED = {}
for name, cfg in REGIONS.items():
    if not cfg.get('geo') and not cfg.get('dep'):
        REGIONS_CLIPPED[name] = cfg
        continue
    print(f'[{name}]')
    REGIONS_CLIPPED[name] = preprocess_region(name, cfg)
print('\nPreprocessing complete.')


## 2. Helper functions

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))


def idw(distances, values, power):
    mask = ~np.isnan(values)
    if mask.sum() == 0:
        return np.nan
    d, v = distances[mask], values[mask]
    if np.any(d < 1e-6):
        return v[d < 1e-6].mean()
    w = 1.0 / (d ** power)
    return np.sum(w * v) / np.sum(w)


def clr(df_elem):
    X = df_elem.astype(float).copy()
    mp = X[X > 0].min().min()
    X = X.mask(X <= 0, mp / 2.0)
    logX = np.log(X)
    return logX.sub(logX.mean(axis=1), axis=0)


## 3. Feature-assignment functions

One function per data source. Each takes the points table and adds columns.


In [ ]:
def make_points(geo_path, dep_path):
    """Positive = REE occurrences; Negative = geochem samples > BUFFER_KM from any occurrence."""
    rng = np.random.default_rng(RANDOM_SEED)
    geo = pd.read_csv(geo_path, low_memory=False).dropna(subset=["latitude", "longitude"]).reset_index(drop=True)
    dep = pd.read_csv(dep_path, low_memory=False).dropna(subset=["Latitude", "Longitude"]).reset_index(drop=True)
    elems = [c for c in ELEMENT_COLS if c in geo.columns]

    pos = pd.DataFrame({"lat": dep["Latitude"].values, "lon": dep["Longitude"].values})
    pos["label"], pos["src"] = 1, "occurrence"

    glat, glon = geo["latitude"].values, geo["longitude"].values
    mind = np.full(len(geo), np.inf)
    for _, d in dep.iterrows():
        mind = np.minimum(mind, haversine_km(d["Latitude"], d["Longitude"], glat, glon))
    enough = geo[elems].notna().sum(axis=1).values >= MIN_ELEMENTS
    far_idx = np.where((mind > BUFFER_KM) & enough)[0]
    if len(far_idx) == 0:
        raise ValueError("no background candidates")
    chosen = rng.choice(far_idx, size=min(N_BACKGROUND, len(far_idx)), replace=False)
    neg = pd.DataFrame({"lat": geo.iloc[chosen]["latitude"].values,
                        "lon": geo.iloc[chosen]["longitude"].values,
                        "label": 0, "src": "background", "geo_row": chosen})
    pts = pd.concat([pos, neg], ignore_index=True)
    return pts, geo, elems


def add_terrane(pts, terrane_path):
    if not terrane_path:
        return pts
    terr = gpd.read_file(terrane_path).to_crs("EPSG:4326")
    keep = [c for c in ["geo_age", "appearance", "crustal_type", "t_name"] if c in terr.columns]
    g = gpd.GeoDataFrame(pts.copy(), geometry=gpd.points_from_xy(pts["lon"], pts["lat"]), crs="EPSG:4326")
    j = gpd.sjoin(g, terr[keep + ["geometry"]], how="left", predicate="within")
    j = j[~j.index.duplicated(keep="first")]
    for c in keep:
        pts[c] = j[c].values
    for c in ["geo_age", "appearance"]:
        if c in pts.columns:
            pts.loc[pts[c] <= -900, c] = np.nan   # Getech -999 = missing
    return pts


def add_structure(pts, struct_path):
    if not struct_path:
        return pts
    s = gpd.read_file(struct_path).to_crs(PROJECTED_CRS)
    su = s.geometry.union_all() if hasattr(s.geometry, "union_all") else s.geometry.unary_union
    g = gpd.GeoDataFrame(pts.copy(), geometry=gpd.points_from_xy(pts["lon"], pts["lat"]),
                         crs="EPSG:4326").to_crs(PROJECTED_CRS)
    dist = g.geometry.apply(lambda x: x.distance(su)).values
    pts["struct_dist_m"] = dist
    dmin, dmax = dist.min(), dist.max()
    pts["struct_proximity"] = 1.0 - (dist - dmin) / (dmax - dmin) if dmax > dmin else 0.0
    return pts


def add_geochem(pts, geo, elems):
    glat, glon = geo["latitude"].values, geo["longitude"].values
    gelem = geo[elems].values
    rows = []
    for _, p in pts.iterrows():
        if p["src"] == "background" and not pd.isna(p.get("geo_row", np.nan)):
            rows.append(geo.iloc[int(p["geo_row"])][elems].values)          # measured
        else:
            dist = haversine_km(p["lat"], p["lon"], glat, glon)
            within = np.where(dist <= SEARCH_RADIUS_KM)[0]
            if len(within) == 0:
                rows.append([np.nan]*len(elems)); continue
            within = within[np.argsort(dist[within])[:K_NEIGHBOURS]]
            rows.append([idw(dist[within], gelem[within, j], IDW_POWER) for j in range(len(elems))])
    pts = pd.concat([pts, pd.DataFrame(rows, columns=elems, index=pts.index)], axis=1)
    return pts


def add_rasters(pts, raster_layers):
    """Sample each raster .tif at every point. Handles CRS reprojection + nodata."""
    if not raster_layers:
        return pts
    import rasterio, pyproj
    for name, path in raster_layers.items():
        with rasterio.open(path) as src:
            if src.crs and src.crs.to_string() != "EPSG:4326":
                tr = pyproj.Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
                xs, ys = tr.transform(pts["lon"].values, pts["lat"].values)
                coords = list(zip(xs, ys))
            else:
                coords = list(zip(pts["lon"], pts["lat"]))
            vals = np.array([v[0] for v in src.sample(coords)], dtype=float)
            if src.nodata is not None:
                vals[vals == src.nodata] = np.nan
        pts[name] = vals
        print(f"    raster '{name}': non-null {np.isfinite(vals).sum()}/{len(vals)}")
    return pts


## 4. Build one region's integrated feature table

Returns the points table, the geochem element list actually used, and the raster
feature names — or `None` if the region is not configured / has too few samples.


In [ ]:
def build_region(name, cfg):
    if not cfg.get("geo") or not cfg.get("dep"):
        print(f"[{name}] geo/dep not set — skipping"); return None
    print(f"[{name}] building integrated features ...")
    pts, geo, elems = make_points(cfg["geo"], cfg["dep"])
    pts = add_terrane(pts, cfg.get("terrane"))
    pts = add_structure(pts, cfg.get("struct"))
    pts = add_geochem(pts, geo, elems)
    pts = add_rasters(pts, cfg.get("raster", {}))

    # keep rows with enough geochem
    pts = pts[pts[elems].notna().sum(axis=1) >= MIN_ELEMENTS].reset_index(drop=True)
    npos, nneg = int((pts["label"]==1).sum()), int((pts["label"]==0).sum())
    print(f"[{name}] points: {npos} positive + {nneg} background; columns: {pts.shape[1]}")
    if npos < 5:
        print(f"[{name}] too few positives — skipping"); return None

    pts.drop(columns=[c for c in ["geo_row"] if c in pts.columns]).to_csv(
        f"{name}_integrated_features.csv", index=False)
    raster_feats = list(cfg.get("raster", {}).keys())
    return {"name": name, "pts": pts, "elems": elems, "raster_feats": raster_feats}


## 5. Feature matrix for PCA/LDA

Geochemistry is compositional → **CLR**; other features (proximity, age, rasters)
are **z-scored** separately, then concatenated. Set `USE_EXTRA=False` to use only
geochemistry (matches the earlier geochem-only analysis).


In [ ]:
# USE_EXTRA = False  # PCA on chemical data only — excludes gravity/magnetic rasters

# def feature_matrix(res):
#     pts, elems, rfeats = res["pts"], res["elems"], res["raster_feats"]
#     y = pts["label"].values

#     # --- geochemistry block: KNN-impute (log) -> CLR -> standardise ---
#     X = pts[elems].astype(float).copy()
#     mp = X[X > 0].min().min(); X = X.mask(X <= 0, mp/2.0)
#     X_imp = pd.DataFrame(np.exp(KNNImputer(n_neighbors=KNN_IMPUTE_K).fit_transform(np.log(X))), columns=elems)
#     Zgeo = StandardScaler().fit_transform(clr(X_imp))
#     blocks, names = [Zgeo], list(elems)

#     # --- extra (non-compositional) block: median-impute -> z-score ---
#     if USE_EXTRA:
#         extra_cols = [c for c in (["struct_proximity", "geo_age"] + rfeats) if c in pts.columns]
#         if extra_cols:
#             E = pts[extra_cols].astype(float).copy()
#             E = E.fillna(E.median())
#             Zextra = StandardScaler().fit_transform(E)
#             blocks.append(Zextra); names += extra_cols

#     Z = np.hstack(blocks)
#     return Z, y, names

## 6. Plotting (scree + PCA biplot/density + LDA)

In [ ]:
# def run_pca_lda(res):
#     Z, y, names = feature_matrix(res)
#     name = res["name"]
#     print(f"\n===== {name} =====  features: {len(names)}  (pos {int((y==1).sum())}, neg {int((y==0).sum())})")

#     pca = PCA(n_components=min(10, Z.shape[1])).fit(Z)
#     sc = pca.transform(Z); var = pca.explained_variance_ratio_
#     print("Explained variance:", ", ".join(f"PC{i}={v*100:.1f}%" for i, v in enumerate(var[:5], 1)))

#     # scree
#     fig, ax = plt.subplots(figsize=(7, 3.5))
#     ax.plot(range(1, len(var)+1), var*100, "o-", color="#8b1a1a")
#     for i, v in enumerate(var, 1):
#         ax.annotate(f"{v*100:.1f}%", (i, v*100), textcoords="offset points", xytext=(0,6), fontsize=7, ha="center")
#     ax.set_xlabel("PC"); ax.set_ylabel("Variance (%)"); ax.set_title(f"{name}: scree")
#     plt.tight_layout(); plt.show()

#     # PCA biplot + density
#     classes = [(0, "Background", "#9bb8cc"), (1, "REE occurrence", "#c0392b")]
#     comp = pca.components_
#     fig = plt.figure(figsize=(13, 6))
#     gs = GridSpec(2, 3, width_ratios=[4,1,4], height_ratios=[1,4], wspace=0.05, hspace=0.05)
#     ax = fig.add_subplot(gs[1,0]); axt = fig.add_subplot(gs[0,0], sharex=ax)
#     axr = fig.add_subplot(gs[1,1], sharey=ax); axb = fig.add_subplot(gs[:,2])
#     for lab, nm, col in classes:
#         m = y == lab
#         ax.scatter(sc[m,0], sc[m,1], s=20, alpha=0.55, label=nm,
#                    facecolor=(col if lab==1 else "none"), edgecolor=col, linewidth=0.6)
#     ax.set_xlabel(f"PC1 ({var[0]*100:.1f}%)"); ax.set_ylabel(f"PC2 ({var[1]*100:.1f}%)")
#     ax.legend(loc="lower right", fontsize=8); ax.set_title(f"(a) {name}", loc="left", fontweight="bold")
#     gx = np.linspace(sc[:,0].min(), sc[:,0].max(), 200); gy = np.linspace(sc[:,1].min(), sc[:,1].max(), 200)
#     for lab, nm, col in classes:
#         m = y == lab
#         axt.plot(gx, gaussian_kde(sc[m,0])(gx), color=col, lw=1.4)
#         axr.plot(gaussian_kde(sc[m,1])(gy), gy, color=col, lw=1.4)
#     axt.axis("off"); axr.axis("off")
#     disp = {"ba_ppm":"Ba","sr_ppm":"Sr"}  # fallback uses ELEM_LABEL / raw name
#     for j, c in enumerate(names):
#         lbl = ELEM_LABEL.get(c, c.replace("_ppm",""))
#         axb.arrow(0,0,comp[0,j],comp[1,j], head_width=0.012, color="black", lw=0.7, length_includes_head=True)
#         axb.text(comp[0,j]*1.13, comp[1,j]*1.13, lbl, fontsize=7, ha="center", va="center")
#     lim = np.abs(comp[:2]).max()*1.3
#     axb.set_xlim(-lim,lim); axb.set_ylim(-lim,lim)
#     axb.axhline(0,color="grey",lw=0.4); axb.axvline(0,color="grey",lw=0.4)
#     axb.set_xlabel("PC1"); axb.set_ylabel("PC2"); axb.set_title("(b)", loc="left", fontweight="bold")
#     plt.show()

#     # PC1 loadings
#     load = pd.Series(comp[0], index=names)
#     print("Top |PC1| loadings:")
#     print(load.abs().sort_values(ascending=False).head(8).round(3))

#     # LDA
#     proj = LinearDiscriminantAnalysis().fit_transform(Z, y).ravel()
#     fig, ax = plt.subplots(figsize=(7, 3.5))
#     for lab, nm, col in [(0,"Background","#9bb8cc"),(1,"REE occurrence","#c0392b")]:
#         m = y == lab; grid = np.linspace(proj.min(), proj.max(), 300)
#         ax.fill_between(grid, gaussian_kde(proj[m])(grid), alpha=0.4, color=col, label=nm)
#         ax.plot(grid, gaussian_kde(proj[m])(grid), color=col, lw=1.5)
#     ax.set_xlabel("LDA axis"); ax.set_ylabel("Density"); ax.set_title(f"{name}: LDA"); ax.legend()
#     plt.tight_layout(); plt.show()

#     auc = cross_val_score(LinearDiscriminantAnalysis(), Z, y, cv=5, scoring="roc_auc")
#     print(f"{name} LDA 5-fold CV AUC: {auc.mean():.3f} +/- {auc.std():.3f}")
#     return {"name": name, "var": var, "auc": auc.mean(), "n_features": len(names)}


## 7. Run all regions

In [ ]:
# summary = {}
# for name, cfg in REGIONS_CLIPPED.items():
#     res = build_region(name, cfg)
#     if res is None:
#         continue
#     summary[name] = run_pca_lda(res)

## 7.5. Geological Structure Distance — Visualisation & Data Export

In [ ]:
# Read the geological structures and boundary files
structures = gpd.read_file('../dataset/Cordilleran_structure.geojson').to_crs('epsg:4326')
boundary   = gpd.read_file('../dataset/Cordilleran.geojson').to_crs('epsg:4326')

# Get the border of the study area
bounds = boundary.total_bounds
xmin, ymin, xmax, ymax = bounds

# Set the cell size of the grid and generate grid point coordinates covering the study area
cell_size = 0.1  # ~10 km in degrees
x = np.arange(xmin, xmax, cell_size)
y = np.arange(ymin, ymax, cell_size)
x_grid, y_grid = np.meshgrid(x, y)
points = np.vstack([x_grid.ravel(), y_grid.ravel()]).T

# Filter grid points within the study area boundary
points_gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(points[:, 0], points[:, 1]), crs='epsg:4326'
)
points_within = points_gdf[points_gdf.within(boundary.unary_union)].copy()
print(f'Grid points within boundary: {len(points_within)}')

# Project to metric CRS (EPSG:3978) for accurate distance calculation in metres
points_proj  = points_within.to_crs(PROJECTED_CRS)
struct_proj  = structures.to_crs(PROJECTED_CRS)
struct_union = struct_proj.geometry.union_all() if hasattr(struct_proj.geometry, 'union_all') \
               else struct_proj.geometry.unary_union

# Calculate distance from each grid point to the nearest structure (metres)
distances = points_proj.geometry.apply(lambda pt: pt.distance(struct_union))

# Normalize and invert: proximity = 1 means closest to a structure
min_d, max_d = distances.min(), distances.max()
struct_proximity = 1 - (distances - min_d) / (max_d - min_d)
points_within['struct_dist_m']    = distances.values
points_within['struct_proximity'] = struct_proximity.values

# Save the data
points_within[['geometry', 'struct_dist_m', 'struct_proximity']].to_csv(
    'Cordilleran_structure_distance.csv', index=False
)
print('Saved: Cordilleran_structure_distance.csv')

# Plot
fig, ax = plt.subplots(figsize=(15, 15))
structures.plot(ax=ax, color=None, edgecolor='black', linewidth=1)
points_plot = points_within.plot(
    ax=ax, column='struct_proximity', cmap='viridis', markersize=3, alpha=0.6
)
boundary.boundary.plot(ax=ax, edgecolor='red', linewidth=1)
import contextily as cx
cx.add_basemap(ax, crs='EPSG:4326', source=cx.providers.Esri.WorldGrayCanvas)
from matplotlib.ticker import FormatStrFormatter
cbar = plt.colorbar(points_plot.collections[0], ax=ax, orientation='vertical')
cbar.set_label('Structure Proximity (1 = closest)')
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
plt.yticks(rotation=90, va='center')
ax.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))
ax.set_title('Geological Structure Distance Distribution')
plt.tight_layout()
plt.show()


## 8. Cross-region summary

In [ ]:
# if summary:
#     tbl = pd.DataFrame({
#         "n_features": {k: v["n_features"] for k, v in summary.items()},
#         "PC1_%": {k: v["var"][0]*100 for k, v in summary.items()},
#         "PC2_%": {k: v["var"][1]*100 for k, v in summary.items()},
#         "LDA_AUC": {k: v["auc"] for k, v in summary.items()},
#     }).round(3)
#     print(tbl.to_string())
# else:
#     print("No regions processed — set file paths in REGIONS.")


## 11. Baseline Random Forest Model & Prospectivity Map

Trains a Random Forest classifier on the Cordilleran integrated features, then
generates a prospectivity map by predicting REE likelihood across a regular grid
covering the study area (bounded by Cordilleran.geojson).

**Outputs:**
- Trained RF model with feature importances
- Prospectivity score raster and GeoTIFF
- Interactive map showing predictions


In [ ]:
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import classification_report, roc_auc_score
# from sklearn.decomposition import PCA
# from sklearn.preprocessing import StandardScaler
# from sklearn.impute import KNNImputer, SimpleImputer
# from rasterio.crs import CRS as RasterCRS
# import rasterio, warnings
# warnings.filterwarnings('ignore')

# cord_df = pd.read_csv("Cordilleran_integrated_features.csv")
# print(f"Loaded {len(cord_df)} samples, {cord_df.shape[1]} columns")

# y_train = cord_df["label"].values

# # ---- Geochemistry: KNN-impute -> log -> CLR -> StandardScaler -> PCA(3) ----
# elem_cols_avail = [c for c in ELEMENT_COLS if c in cord_df.columns]
# X_elem = cord_df[elem_cols_avail].astype(float).copy()
# mp = X_elem[X_elem > 0].min().min()
# X_elem = X_elem.mask(X_elem <= 0, mp / 2.0)
# elem_imputer = KNNImputer(n_neighbors=KNN_IMPUTE_K)
# X_elem_imp = pd.DataFrame(
#     np.exp(elem_imputer.fit_transform(np.log(X_elem))),
#     columns=elem_cols_avail
# )
# geo_scaler = StandardScaler()
# X_clr_scaled = geo_scaler.fit_transform(clr(X_elem_imp))
# geo_pca = PCA(n_components=5, random_state=RANDOM_SEED)
# X_pca = geo_pca.fit_transform(X_clr_scaled)
# var = geo_pca.explained_variance_ratio_
# print("Geochemistry PCA: " + "  ".join(f"PC{i+1}={v*100:.1f}%" for i, v in enumerate(var)) + f"  (total={sum(var)*100:.1f}%)")

# # ---- Extra features (struct, terrane — whatever is in CSV) ----
# skip = set(["lat", "lon", "label", "src", "geo_row", "t_name"] + elem_cols_avail)
# extra_cols = [c for c in cord_df.columns
#               if c not in skip and cord_df[c].dtype in ["float64", "int64", "float32", "int32"]]
# extra_imputer = SimpleImputer(strategy="median")
# X_extra = extra_imputer.fit_transform(cord_df[extra_cols].astype(float).values)

# # ---- Combine PC components + extra, train RF ----
# X_train = np.hstack([X_pca, X_extra])
# feature_names = [f"PC{i+1}" for i in range(geo_pca.n_components_)] + extra_cols
# print(f"Features ({len(feature_names)}): {feature_names}")

# rf_model = RandomForestClassifier(
#     n_estimators=200, max_depth=15, min_samples_split=5, min_samples_leaf=2,
#     random_state=RANDOM_SEED, n_jobs=-1, class_weight="balanced"
# )
# rf_model.fit(X_train, y_train)

# y_pred = rf_model.predict(X_train)
# y_pred_proba = rf_model.predict_proba(X_train)[:, 1]
# print(f"\n=== RF Training Results ===")
# print(f"Accuracy: {(y_pred == y_train).mean():.3f}  |  ROC-AUC: {roc_auc_score(y_train, y_pred_proba):.3f}")
# print(classification_report(y_train, y_pred, target_names=["Background", "REE"]))

# feat_imp = pd.Series(rf_model.feature_importances_, index=feature_names).sort_values(ascending=False)
# print("Feature Importances:")
# print(feat_imp.round(4))
# fig, ax = plt.subplots(figsize=(8, 4))
# feat_imp.plot(kind="barh", ax=ax, color="steelblue")
# ax.set_xlabel("Importance"); ax.set_title(f"RF Feature Importances (PC1~PC{geo_pca.n_components_} + spatial)")
# plt.tight_layout(); plt.show()


In [ ]:
# import rasterio, pyproj
# from scipy.interpolate import griddata as scipy_griddata

# # 纯边界网格
# boundary_gdf = gpd.read_file("../dataset/Cordilleran.geojson").to_crs("EPSG:4326")
# bounds = boundary_gdf.total_bounds

# grid_spacing = 0.2  # ~20 km
# xs = np.arange(bounds[0], bounds[2] + grid_spacing, grid_spacing)
# ys = np.arange(bounds[1], bounds[3] + grid_spacing, grid_spacing)
# xx, yy = np.meshgrid(xs, ys)
# grid_gdf = gpd.GeoDataFrame(
#     {"grid_id": range(len(xx.ravel()))},
#     geometry=gpd.points_from_xy(xx.ravel(), yy.ravel()),
#     crs="EPSG:4326"
# )
# grid_gdf = gpd.sjoin(grid_gdf, boundary_gdf, how="inner", predicate="within")
# grid_gdf = grid_gdf[["grid_id", "geometry"]].drop_duplicates(subset=["grid_id"]).reset_index(drop=True)
# print(f"Grid points within study area: {len(grid_gdf)}")

# grid_lons = grid_gdf.geometry.x.values
# grid_lats = grid_gdf.geometry.y.values
# query_pts = np.column_stack([grid_lons, grid_lats])

# grid_features = pd.DataFrame(index=range(len(grid_gdf)))

# # ---- 地球化学：scipy griddata 向量化插值（cubic平滑，比linear更好）----
# geo_df = pd.read_csv("../dataset/Geochemistry_Cordilleran_Clip.csv", low_memory=False).dropna(subset=["latitude", "longitude"])
# elem_cols_grid = [c for c in ELEMENT_COLS if c in geo_df.columns]
# geo_pts = np.column_stack([geo_df["longitude"].values, geo_df["latitude"].values])

# elem_interp = {}
# for elem in elem_cols_grid:
#     vals = geo_df[elem].values.astype(float)
#     mask = ~np.isnan(vals)
#     if mask.sum() > 3:
#         # 先尝试cubic（更平滑），如果失败回退到linear
#         try:
#             elem_interp[elem] = scipy_griddata(geo_pts[mask], vals[mask], query_pts, method="cubic")
#         except:
#             elem_interp[elem] = scipy_griddata(geo_pts[mask], vals[mask], query_pts, method="linear")
#     else:
#         elem_interp[elem] = np.full(len(grid_lons), np.nan)

# elem_df = pd.DataFrame(elem_interp, index=range(len(grid_gdf)))
# print(f"  地球化学插值完成: {elem_df.shape}，非空率(Ba): {elem_df[elem_cols_grid[0]].notna().mean()*100:.0f}%")

# # ---- 对插值结果做 CLR -> geo_scaler -> geo_pca，得到真实 PC 值 ----
# enough = elem_df.notna().sum(axis=1) >= MIN_ELEMENTS
# X_e = elem_df.copy()
# min_pos = X_e[X_e > 0].min().min()
# X_e = X_e.mask(X_e <= 0, min_pos / 2.0)
# for col in elem_cols_grid:
#     med = X_e[col].median()
#     X_e[col] = X_e[col].fillna(med if pd.notna(med) else 0.0)

# X_clr_grid = clr(X_e)
# X_pca_grid = geo_pca.transform(geo_scaler.transform(X_clr_grid))
# X_pca_grid[~enough.values] = np.nan  # 化学覆盖不足的格子标为 NaN

# for i in range(geo_pca.n_components_):
#     grid_features[f"PC{i+1}"] = X_pca_grid[:, i]

# non_null = (~np.isnan(X_pca_grid[:, 0])).sum()
# print(f"  PC 特征: {geo_pca.n_components_} 个，{non_null}/{len(grid_gdf)} 格子有真实值，其余用空间邻近IDW填充")

# # ---- 栅格采样（向量化）----
# for feat_name, raster_path in REGIONS["Cordilleran"]["raster"].items():
#     with rasterio.open(raster_path) as src:
#         if src.crs and src.crs.to_string() != "EPSG:4326":
#             tr = pyproj.Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
#             xs_r, ys_r = tr.transform(grid_lons, grid_lats)
#             coords = list(zip(xs_r, ys_r))
#         else:
#             coords = list(zip(grid_lons, grid_lats))
#         vals = np.array([v[0] for v in src.sample(coords)], dtype=float)
#         if src.nodata is not None:
#             vals[vals == src.nodata] = np.nan
#     grid_features[feat_name] = vals
#     print(f"  raster '{feat_name}': non-null {np.isfinite(vals).sum()}/{len(vals)}")

# # ---- 构造邻近度 ----
# struct = gpd.read_file("../dataset/Cordilleran_structure.geojson").to_crs(PROJECTED_CRS)
# su = struct.geometry.union_all() if hasattr(struct.geometry, "union_all") else struct.geometry.unary_union
# dist_s = grid_gdf.to_crs(PROJECTED_CRS).geometry.apply(lambda x: x.distance(su)).values
# dmin, dmax = dist_s.min(), dist_s.max()
# grid_features["struct_proximity"] = 1.0 - (dist_s - dmin) / (dmax - dmin) if dmax > dmin else 0.0
# print("  struct_proximity: done")

# # ---- 地体特征 ----
# terr = gpd.read_file("../dataset/Cordilleran_Terrane.geojson").to_crs("EPSG:4326")
# keep = [c for c in ["geo_age", "appearance"] if c in terr.columns]
# j = gpd.sjoin(grid_gdf, terr[keep + ["geometry"]], how="left", predicate="within")
# j = j[~j.index.duplicated(keep="first")]
# for col in keep:
#     vals_t = j[col].values.astype(float)
#     if col in ["geo_age", "appearance"]:
#         vals_t[vals_t <= -900] = np.nan
#     grid_features[col] = vals_t
# print(f"  terrane: done ({keep})")

# print(f"\nGrid features shape: {grid_features.shape}")


In [ ]:
# from scipy.interpolate import griddata
# from rasterio.transform import Affine
# from scipy.spatial import cKDTree

# # PC 值：有真实插值就用真实值，没有（化学覆盖不足）才用空间邻近IDW填充
# pc_cols = [f"PC{i+1}" for i in range(geo_pca.n_components_)]
# pc_medians = np.median(X_pca, axis=0)
# X_pc_grid = grid_features[pc_cols].values.copy()

# # 为有真实值的点构建KD-tree，进行空间邻近IDW填充
# valid_pts_mask = ~np.isnan(X_pc_grid[:, 0])
# print(f"  PC值有真实覆盖的格子: {valid_pts_mask.sum()}/{len(grid_gdf)}")

# if valid_pts_mask.sum() > 3:
#     # 获取有真实PC值的点的坐标
#     grid_proj = grid_gdf.to_crs(PROJECTED_CRS)
#     xs_all = grid_proj.geometry.x.values
#     ys_all = grid_proj.geometry.y.values
#     valid_pts = np.column_stack([xs_all[valid_pts_mask], ys_all[valid_pts_mask]])
#     tree = cKDTree(valid_pts)
    
#     for i, col in enumerate(pc_cols):
#         nan_mask = np.isnan(X_pc_grid[:, i])
#         if nan_mask.sum() > 0:
#             nan_indices = np.where(nan_mask)[0]
#             nan_pts = np.column_stack([xs_all[nan_indices], ys_all[nan_indices]])
            
#             # 查询最近的K个有效点
#             k_neighbors = min(5, valid_pts_mask.sum())
#             dists, neighbors_idx = tree.query(nan_pts, k=k_neighbors)
            
#             # IDW插值（距离越近权重越大）
#             for j, (d, neighbor_indices) in enumerate(zip(dists, neighbors_idx)):
#                 if np.min(d) < 1e10:  # 合理距离范围
#                     weights = 1.0 / (d + 1e-8)
#                     valid_indices = np.where(valid_pts_mask)[0][neighbor_indices]
#                     X_pc_grid[nan_indices[j], i] = np.sum(weights * X_pc_grid[valid_indices, i]) / np.sum(weights)
#                 else:
#                     X_pc_grid[nan_indices[j], i] = pc_medians[i]
#             print(f"  {col}: {nan_mask.sum()} 个格子用空间邻近IDW填充（K=5，距离加权）")
# else:
#     # 如果有效点太少，回退到全局中位数
#     for i, col in enumerate(pc_cols):
#         nan_mask = np.isnan(X_pc_grid[:, i])
#         X_pc_grid[nan_mask, i] = pc_medians[i]
#         if nan_mask.sum() > 0:
#             print(f"  {col}: {nan_mask.sum()} 个格子用全局中位数填充（有效点不足）")

# # 空间特征（栅格 + 构造 + 地体）
# # 确保 grid_features 包含 struct_dist_m（若 cell 20 未计算）
# if "struct_dist_m" in extra_cols and "struct_dist_m" not in grid_features.columns:
#     _struct = gpd.read_file("../dataset/Cordilleran_structure.geojson").to_crs(PROJECTED_CRS)
#     _su = _struct.geometry.union_all() if hasattr(_struct.geometry,"union_all") else _struct.geometry.unary_union
#     print("计算 grid struct_dist_m...")
#     grid_features["struct_dist_m"] = np.array([pt.distance(_su) for pt in grid_gdf.to_crs(PROJECTED_CRS).geometry])

# X_extra_grid = extra_imputer.transform(
#     grid_features.reindex(columns=extra_cols).astype(float).values
# )

# X_grid_full = np.hstack([X_pc_grid, X_extra_grid])
# grid_proba = rf_model.predict_proba(X_grid_full)[:, 1]
# grid_gdf["prospectivity"] = grid_proba

# print(f"\nProspectivity: mean={grid_proba.mean():.3f}  std={grid_proba.std():.3f}  min={grid_proba.min():.3f}  max={grid_proba.max():.3f}")

# grid_gdf.to_file("Cordilleran_prospectivity_points.geojson", driver="GeoJSON")
# print("Saved: Cordilleran_prospectivity_points.geojson")

# # 插值为栅格用于可视化
# grid_proj = grid_gdf.to_crs(PROJECTED_CRS)
# xs_p, ys_p = grid_proj.geometry.x.values, grid_proj.geometry.y.values
# x_min, y_min, x_max, y_max = xs_p.min(), ys_p.min(), xs_p.max(), ys_p.max()
# xi = np.arange(x_min, x_max, 5000)
# yi = np.arange(y_min, y_max, 5000)
# Xi, Yi = np.meshgrid(xi, yi)
# Z = griddata(np.column_stack([xs_p, ys_p]), grid_proba, (Xi, Yi), method="linear")
# # 裁剪到研究区边界（去掉边界外的颜色）
# from rasterio.features import geometry_mask
# _bnd = boundary_gdf.to_crs(PROJECTED_CRS)
# _transform = Affine.translation(x_min, y_max) * Affine.scale(5000, -5000)
# _mask = geometry_mask(_bnd.geometry, out_shape=Z.shape, transform=_transform, invert=True)
# Z = np.where(_mask, Z, np.nan)


# transform = Affine.translation(x_min, y_max) * Affine.scale(5000, -5000)
# with rasterio.open(
#     "Cordilleran_prospectivity.tif", "w",
#     driver="GTiff", height=Z.shape[0], width=Z.shape[1], count=1, dtype=Z.dtype,
#     crs=RasterCRS.from_epsg(3978), transform=transform, nodata=np.nan
# ) as dst:
#     dst.write(Z, 1)
# print("Saved: Cordilleran_prospectivity.tif")

# # 画图 — 离散分级颜色
# import matplotlib.colors as mcolors
# import matplotlib.patches as mpatches

# breaks = [0.0,
#           round(float(np.percentile(grid_proba, 25)), 3),
#           round(float(np.percentile(grid_proba, 50)), 3),
#           round(float(np.percentile(grid_proba, 90)), 3),
#           1.0]
# print(f"分级断点: {breaks}")

# colors_cls = ['#1a6faf', '#42c4e8', '#f5e21a', '#d73027']
# cmap_cls   = mcolors.ListedColormap(colors_cls)
# norm_cls   = mcolors.BoundaryNorm(breaks, len(colors_cls))

# fig, ax = plt.subplots(figsize=(14, 11))
# ax.imshow(np.ma.masked_invalid(Z), extent=[x_min, x_max, y_min, y_max],
#           origin="upper", cmap=cmap_cls, norm=norm_cls, aspect="auto", alpha=0.9)

# boundary_gdf.to_crs(PROJECTED_CRS).boundary.plot(ax=ax, edgecolor="black", linewidth=1.5)

# dep = pd.read_csv("../dataset/CarbonatitehostREE_Cordilleran_Clip.csv", low_memory=False)
# gpd.GeoDataFrame(dep, geometry=gpd.points_from_xy(dep["Longitude"], dep["Latitude"]),
#                  crs="EPSG:4326").to_crs(PROJECTED_CRS).plot(
#     ax=ax, color="red", markersize=80, marker="*", alpha=0.45, zorder=5)

# legend_patches = [
#     mpatches.Patch(facecolor=colors_cls[i], edgecolor="grey", linewidth=0.5,
#                    label=f"{breaks[i]:.3f} - {breaks[i+1]:.3f}")
#     for i in range(len(colors_cls))
# ]
# legend_patches.append(
#     plt.Line2D([0], [0], marker="*", color="w", markerfacecolor="red",
#                markersize=12, alpha=0.7, label="Known REE Occurrences")
# )
# ax.legend(handles=legend_patches, loc="lower right", fontsize=10,
#           title="Prospectivity Score", title_fontsize=10,
#           framealpha=0.9, edgecolor="grey")

# ax.set_xlabel("Easting (m)", fontsize=11)
# ax.set_ylabel("Northing (m)", fontsize=11)
# ax.set_title("Cordilleran REE Prospectivity (RF: PC1-5 + Gravity/Magnetic + Structure)",
#              fontsize=13, fontweight="bold")
# ax.grid(True, alpha=0.3)
# plt.tight_layout(); plt.show()

# high = grid_gdf[grid_gdf["prospectivity"] > np.percentile(grid_proba, 90)]
# print(f"高成矿概率区域（前10%）: {len(high)} 个格子，平均分={high['prospectivity'].mean():.3f}")


## 12. RF Model v2: Raster + Geology Features Only (No PCA)

Test RF feasibility using only **raster + geological features** (20-5 features),
bypassing PCA dimensionality reduction to see raw feature importance and
compare performance vs. the PCA-based model above.

**Features used:**
- All raster layers (gravity, magnetic, etc.)
- Structural proximity
- Terrane age & appearance



In [ ]:
# from sklearn.model_selection import cross_val_score, cross_validate
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
# import warnings
# warnings.filterwarnings('ignore')

# # 加载数据
# cord_df_v2 = pd.read_csv("Cordilleran_integrated_features.csv")
# print(f"Loaded {len(cord_df_v2)} samples, {cord_df_v2.shape[1]} columns")

# y_train_v2 = cord_df_v2["label"].values

# # 选择特征：raster + geology（不包括geochemistry和PCA）
# skip_cols = set(["lat", "lon", "label", "src", "geo_row", "t_name",
#                 "struct_proximity"]  # 归一化依赖点集，训练/预测不一致，改用 struct_dist_m
#                + ELEMENT_COLS)
# candidate_cols = [c for c in cord_df_v2.columns if c not in skip_cols and cord_df_v2[c].dtype in ["float64", "int64", "float32", "int32"]]

# # 分离raster特征和geology特征
# raster_features = [c for c in REGIONS["Cordilleran"]["raster"].keys() if c in candidate_cols]
# geology_features = [c for c in candidate_cols if c not in raster_features]

# print(f"\n=== Feature Selection (No PCA) ===")
# print(f"Raster features ({len(raster_features)}): {raster_features}")
# print(f"Geology features ({len(geology_features)}): {geology_features}")
# print(f"Total features: {len(raster_features) + len(geology_features)}")

# X_features_v2 = raster_features + geology_features

# # 数据预处理：中位数填充
# imputer_v2 = SimpleImputer(strategy="median")
# X_train_v2 = imputer_v2.fit_transform(cord_df_v2[X_features_v2].astype(float).values)

# print(f"\n非空率:")
# for feat in X_features_v2:
#     non_null = cord_df_v2[feat].notna().sum()
#     print(f"  {feat}: {non_null}/{len(cord_df_v2)} ({non_null/len(cord_df_v2)*100:.1f}%)")

# # 训练RF模型（不用PCA）
# rf_model_v2 = RandomForestClassifier(
#     n_estimators=200, max_depth=15, min_samples_split=5, min_samples_leaf=2,
#     random_state=RANDOM_SEED, n_jobs=-1, class_weight="balanced"
# )
# rf_model_v2.fit(X_train_v2, y_train_v2)

# # 训练集评估
# y_pred_v2 = rf_model_v2.predict(X_train_v2)
# y_pred_proba_v2 = rf_model_v2.predict_proba(X_train_v2)[:, 1]
# train_auc_v2 = roc_auc_score(y_train_v2, y_pred_proba_v2)
# train_acc_v2 = (y_pred_v2 == y_train_v2).mean()

# print(f"\n=== RF v2 Training Results (Raster + Geology Only) ===")
# print(f"Accuracy: {train_acc_v2:.3f}  |  ROC-AUC: {train_auc_v2:.3f}")
# print(classification_report(y_train_v2, y_pred_v2, target_names=["Background", "REE"]))

# # 5折交叉验证
# cv_scores_v2 = cross_validate(
#     rf_model_v2, X_train_v2, y_train_v2, cv=5,
#     scoring={"accuracy": "accuracy", "roc_auc": "roc_auc", "precision": "precision", "recall": "recall"},
#     n_jobs=-1
# )
# print(f"\n=== 5-Fold Cross-Validation (Raster + Geology Only) ===")
# print(f"Accuracy:  {cv_scores_v2['test_accuracy'].mean():.3f} +/- {cv_scores_v2['test_accuracy'].std():.3f}")
# print(f"ROC-AUC:   {cv_scores_v2['test_roc_auc'].mean():.3f} +/- {cv_scores_v2['test_roc_auc'].std():.3f}")
# print(f"Precision: {cv_scores_v2['test_precision'].mean():.3f} +/- {cv_scores_v2['test_precision'].std():.3f}")
# print(f"Recall:    {cv_scores_v2['test_recall'].mean():.3f} +/- {cv_scores_v2['test_recall'].std():.3f}")

# # 特征重要性
# feat_imp_v2 = pd.Series(rf_model_v2.feature_importances_, index=X_features_v2).sort_values(ascending=False)
# print("\n=== Feature Importances (Top 15) ===")
# print(feat_imp_v2.head(15).round(4))

# fig, ax = plt.subplots(figsize=(10, 6))
# feat_imp_v2.head(15).plot(kind="barh", ax=ax, color="coral")
# ax.set_xlabel("Importance")
# ax.set_title("RF Feature Importances (Raster + Geology, No PCA)")
# plt.tight_layout(); plt.show()

# # 对比PCA版本
# print(f"\n=== PCA vs No-PCA 模型对比 ===")
# print(f"PCA版本 (5 PC components + spatial):  AUC={roc_auc_score(y_train, y_pred_proba):.3f}")
# print(f"无PCA版本 (Raster + Geology):         AUC={train_auc_v2:.3f}")


In [ ]:
# # 用v2模型生成prospectivity地图
# print("生成v2模型prospectivity地图...")

# # 从grid_features中提取相同的特征
# # 确保 grid_features 包含 struct_dist_m（若 cell 20 未计算）
# if "struct_dist_m" in X_features_v2 and "struct_dist_m" not in grid_features.columns:
#     _struct = gpd.read_file("../dataset/Cordilleran_structure.geojson").to_crs(PROJECTED_CRS)
#     _su = _struct.geometry.union_all() if hasattr(_struct.geometry,"union_all") else _struct.geometry.unary_union
#     print("计算 grid struct_dist_m...")
#     grid_features["struct_dist_m"] = np.array([pt.distance(_su) for pt in grid_gdf.to_crs(PROJECTED_CRS).geometry])

# X_grid_v2 = imputer_v2.transform(
#     grid_features.reindex(columns=X_features_v2).astype(float).values
# )

# # 预测
# grid_proba_v2 = rf_model_v2.predict_proba(X_grid_v2)[:, 1]
# grid_gdf["prospectivity_v2"] = grid_proba_v2

# print(f"\nProspectivity (v2): mean={grid_proba_v2.mean():.3f}  std={grid_proba_v2.std():.3f}  min={grid_proba_v2.min():.3f}  max={grid_proba_v2.max():.3f}")

# grid_gdf.to_file("Cordilleran_prospectivity_v2_points.geojson", driver="GeoJSON")
# print("Saved: Cordilleran_prospectivity_v2_points.geojson")

# # 插值为栅格
# grid_proj_v2 = grid_gdf.to_crs(PROJECTED_CRS)
# xs_p_v2, ys_p_v2 = grid_proj_v2.geometry.x.values, grid_proj_v2.geometry.y.values
# x_min_v2, y_min_v2, x_max_v2, y_max_v2 = xs_p_v2.min(), ys_p_v2.min(), xs_p_v2.max(), ys_p_v2.max()
# xi_v2 = np.arange(x_min_v2, x_max_v2, 5000)
# yi_v2 = np.arange(y_min_v2, y_max_v2, 5000)
# Xi_v2, Yi_v2 = np.meshgrid(xi_v2, yi_v2)
# Z_v2 = griddata(np.column_stack([xs_p_v2, ys_p_v2]), grid_proba_v2, (Xi_v2, Yi_v2), method="linear")
# # 裁剪到研究区边界
# from rasterio.features import geometry_mask
# _bnd = boundary_gdf.to_crs(PROJECTED_CRS)
# _transform = Affine.translation(x_min_v2, y_max_v2) * Affine.scale(5000, -5000)
# _mask = geometry_mask(_bnd.geometry, out_shape=Z_v2.shape, transform=_transform, invert=True)
# Z_v2 = np.where(_mask, Z_v2, np.nan)


# transform_v2 = Affine.translation(x_min_v2, y_max_v2) * Affine.scale(5000, -5000)
# with rasterio.open(
#     "Cordilleran_prospectivity_v2.tif", "w",
#     driver="GTiff", height=Z_v2.shape[0], width=Z_v2.shape[1], count=1, dtype=Z_v2.dtype,
#     crs=RasterCRS.from_epsg(3978), transform=transform_v2, nodata=np.nan
# ) as dst:
#     dst.write(Z_v2, 1)
# print("Saved: Cordilleran_prospectivity_v2.tif")

# # 绘制v2地图
# breaks_v2 = [0.0,
#              round(float(np.percentile(grid_proba_v2, 25)), 3),
#              round(float(np.percentile(grid_proba_v2, 80)), 3),
#              round(float(np.percentile(grid_proba_v2, 90)), 3),
#              1.0]
# print(f"分级断点 (v2): {breaks_v2}")

# colors_cls_v2 = ['#1a6faf', '#42c4e8', '#f5e21a', '#d73027']
# cmap_cls_v2   = mcolors.ListedColormap(colors_cls_v2)
# norm_cls_v2   = mcolors.BoundaryNorm(breaks_v2, len(colors_cls_v2))

# fig, ax = plt.subplots(figsize=(14, 11))
# ax.imshow(np.ma.masked_invalid(Z_v2), extent=[x_min_v2, x_max_v2, y_min_v2, y_max_v2],
#           origin="upper", cmap=cmap_cls_v2, norm=norm_cls_v2, aspect="auto", alpha=0.9)

# boundary_gdf.to_crs(PROJECTED_CRS).boundary.plot(ax=ax, edgecolor="black", linewidth=1.5)

# dep_v2 = pd.read_csv("../dataset/CarbonatitehostREE_Cordilleran_Clip.csv", low_memory=False)
# gpd.GeoDataFrame(dep_v2, geometry=gpd.points_from_xy(dep_v2["Longitude"], dep_v2["Latitude"]),
#                  crs="EPSG:4326").to_crs(PROJECTED_CRS).plot(
#     ax=ax, color="red", markersize=80, marker="*", alpha=0.45, zorder=5)

# legend_patches_v2 = [
#     mpatches.Patch(facecolor=colors_cls_v2[i], edgecolor="grey", linewidth=0.5,
#                    label=f"{breaks_v2[i]:.3f} - {breaks_v2[i+1]:.3f}")
#     for i in range(len(colors_cls_v2))
# ]
# legend_patches_v2.append(
#     plt.Line2D([0], [0], marker="*", color="w", markerfacecolor="red",
#                markersize=12, alpha=0.7, label="Known REE Occurrences")
# )
# ax.legend(handles=legend_patches_v2, loc="lower right", fontsize=10,
#           title="Prospectivity Score (v2)", title_fontsize=10,
#           framealpha=0.9, edgecolor="grey")

# ax.set_xlabel("Easting (m)", fontsize=11)
# ax.set_ylabel("Northing (m)", fontsize=11)
# ax.set_title("Cordilleran REE Prospectivity - RF v2 (Raster + Geology, No PCA)",
#              fontsize=13, fontweight="bold")
# ax.grid(True, alpha=0.3)
# plt.tight_layout(); plt.show()

# high_v2 = grid_gdf[grid_gdf["prospectivity_v2"] > np.percentile(grid_proba_v2, 90)]
# print(f"\n高成矿概率区域（前10%，v2）: {len(high_v2)} 个格子，平均分={high_v2['prospectivity_v2'].mean():.3f}")

# # 对比两个版本的prospectivity
# print(f"\n=== 两个版本prospectivity对比 ===")
# corr = np.corrcoef(grid_proba, grid_proba_v2)[0, 1]
# print(f"Pearson相关系数: {corr:.4f}")
# print(f"v1 (PCA)   mean={grid_proba.mean():.4f}  std={grid_proba.std():.4f}")
# print(f"v2 (无PCA) mean={grid_proba_v2.mean():.4f}  std={grid_proba_v2.std():.4f}")

# # 绘制对比散点图
# fig, ax = plt.subplots(figsize=(8, 8))
# ax.scatter(grid_proba, grid_proba_v2, alpha=0.3, s=10, color="steelblue")
# ax.plot([0, 1], [0, 1], "r--", lw=2, label="y=x")
# ax.set_xlabel("Prospectivity (v1: PCA)", fontsize=11)
# ax.set_ylabel("Prospectivity (v2: Raster+Geology)", fontsize=11)
# ax.set_title(f"RF v1 vs v2 Prospectivity Comparison (r={corr:.3f})", fontsize=12, fontweight="bold")
# ax.legend(fontsize=10)
# ax.grid(True, alpha=0.3)
# plt.tight_layout(); plt.show()


## 15. 优化网格分辨率 - 更密集的预测网格 

现在不使用PCA，所有特征都是"高质量"的（无missing value插补问题）。
可以大幅增加网格密度，改进prospectivity map的精度。



In [ ]:
# RF v2 高精度成矿预测图（只用 raster + geology，无 PCA，无地球化学）
import rasterio, pyproj
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from scipy.interpolate import griddata
from rasterio.transform import Affine

# ---- 1. 在 Cordilleran 边界内生成细网格 ----
boundary_gdf = gpd.read_file("../dataset/Cordilleran.geojson").to_crs("EPSG:4326")
bounds = boundary_gdf.total_bounds

grid_spacing_deg = 0.05   # ~5km，可调细（0.02°~2km）
xs = np.arange(bounds[0], bounds[2] + grid_spacing_deg, grid_spacing_deg)
ys = np.arange(bounds[1], bounds[3] + grid_spacing_deg, grid_spacing_deg)
xx, yy = np.meshgrid(xs, ys)
fine_gdf = gpd.GeoDataFrame(
    {"grid_id": range(len(xx.ravel()))},
    geometry=gpd.points_from_xy(xx.ravel(), yy.ravel()),
    crs="EPSG:4326"
)
fine_gdf = gpd.sjoin(fine_gdf, boundary_gdf, how="inner", predicate="within")
fine_gdf = fine_gdf[["grid_id", "geometry"]].drop_duplicates(subset=["grid_id"]).reset_index(drop=True)
fine_lons = fine_gdf.geometry.x.values
fine_lats = fine_gdf.geometry.y.values
print(f"细网格: {len(fine_gdf)} 个格子（{grid_spacing_deg}°间距，{grid_spacing_deg*111:.0f}km）")

fine_feats = pd.DataFrame(index=range(len(fine_gdf)))

# ---- 2. 采样所有 raster（向量化）----
for feat_name, raster_path in REGIONS["Cordilleran"]["raster"].items():
    with rasterio.open(raster_path) as src:
        if src.crs and src.crs.to_string() != "EPSG:4326":
            tr = pyproj.Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
            xs_r, ys_r = tr.transform(fine_lons, fine_lats)
            coords = list(zip(xs_r, ys_r))
        else:
            coords = list(zip(fine_lons, fine_lats))
        vals = np.array([v[0] for v in src.sample(coords)], dtype=float)
        if src.nodata is not None:
            vals[vals == src.nodata] = np.nan
    fine_feats[feat_name] = vals
print(f"  rasters 采样完成（{len(REGIONS['Cordilleran']['raster'])} 层）")

# ---- 3. 构造距离（原始米数，尺度固定不受点集影响）----
if "struct_dist_m" in X_features_v2:
    struct = gpd.read_file("../dataset/Cordilleran_structure.geojson").to_crs(PROJECTED_CRS)
    su = struct.geometry.union_all() if hasattr(struct.geometry, "union_all") else struct.geometry.unary_union
    dist_s = fine_gdf.to_crs(PROJECTED_CRS).geometry.apply(lambda x: x.distance(su)).values
    fine_feats["struct_dist_m"] = dist_s
    print("  struct_dist_m 采样完成")

# ---- 4. 地体特征（spatial join）----
terrane_feats = [c for c in ["geo_age", "appearance"] if c in X_features_v2]
if terrane_feats:
    terr = gpd.read_file("../dataset/Cordilleran_Terrane.geojson").to_crs("EPSG:4326")
    j = gpd.sjoin(fine_gdf, terr[terrane_feats + ["geometry"]], how="left", predicate="within")
    j = j[~j.index.duplicated(keep="first")]
    for col in terrane_feats:
        vals_t = j[col].values.astype(float)
        vals_t[vals_t <= -900] = np.nan
        fine_feats[col] = vals_t
    print(f"  地体特征采样完成 {terrane_feats}")

# ---- 5. 预测 ----
X_fine = imputer_v2.transform(
    fine_feats.reindex(columns=X_features_v2).astype(float).values
)
fine_proba = rf_model_v2.predict_proba(X_fine)[:, 1]
fine_gdf["prospectivity"] = fine_proba
print(f"\nProspectivity: mean={fine_proba.mean():.3f}  std={fine_proba.std():.3f}  min={fine_proba.min():.3f}  max={fine_proba.max():.3f}")

# ---- 6. 保存 ----
fine_gdf.to_file("Cordilleran_prospectivity_v2_fine.geojson", driver="GeoJSON")
print("Saved: Cordilleran_prospectivity_v2_fine.geojson")

# ---- 7. 画图 ----
breaks = [0.0,
          round(float(np.percentile(fine_proba, 25)), 3),
          round(float(np.percentile(fine_proba, 50)), 3),
          round(float(np.percentile(fine_proba, 90)), 3),
          1.0]
print(f"分级断点: {breaks}")

colors_cls = ["#1a6faf", "#42c4e8", "#f5e21a", "#d73027"]
cmap_cls   = mcolors.ListedColormap(colors_cls)
norm_cls   = mcolors.BoundaryNorm(breaks, len(colors_cls))

# 把散点插值为规则栅格用于 imshow
fine_proj = fine_gdf.to_crs(PROJECTED_CRS)
xs_p, ys_p = fine_proj.geometry.x.values, fine_proj.geometry.y.values
x_min, y_min, x_max, y_max = xs_p.min(), ys_p.min(), xs_p.max(), ys_p.max()
res = int(grid_spacing_deg * 111000)  # 像素大小跟网格间距匹配
xi = np.arange(x_min, x_max, res)
yi = np.arange(y_min, y_max, res)
Xi, Yi = np.meshgrid(xi, yi)
Z = griddata(np.column_stack([xs_p, ys_p]), fine_proba, (Xi, Yi), method="linear")
# 裁剪到研究区边界（去掉边界外的颜色）
from rasterio.features import geometry_mask
_bnd = boundary_gdf.to_crs(PROJECTED_CRS)
_transform = Affine.translation(x_min, y_max) * Affine.scale(res, -res)
_mask = geometry_mask(_bnd.geometry, out_shape=Z.shape, transform=_transform, invert=True)
Z = np.where(_mask, Z, np.nan)


fig, ax = plt.subplots(figsize=(14, 11))
ax.imshow(np.ma.masked_invalid(Z), extent=[x_min, x_max, y_min, y_max],
          origin="upper", cmap=cmap_cls, norm=norm_cls, aspect="auto", alpha=0.9)

boundary_gdf.to_crs(PROJECTED_CRS).boundary.plot(ax=ax, edgecolor="black", linewidth=1.5)

dep = pd.read_csv("../dataset/CarbonatitehostREE_Cordilleran_Clip.csv", low_memory=False)
gpd.GeoDataFrame(dep, geometry=gpd.points_from_xy(dep["Longitude"], dep["Latitude"]),
                 crs="EPSG:4326").to_crs(PROJECTED_CRS).plot(
    ax=ax, color="red", markersize=80, marker="*", alpha=0.45, zorder=5)

legend_patches = [
    mpatches.Patch(facecolor=colors_cls[i], edgecolor="grey", linewidth=0.5,
                   label=f"{breaks[i]:.3f} - {breaks[i+1]:.3f}")
    for i in range(len(colors_cls))
]
legend_patches.append(
    plt.Line2D([0], [0], marker="*", color="w", markerfacecolor="red",
               markersize=12, alpha=0.7, label="Known REE Occurrences")
)
ax.legend(handles=legend_patches, loc="lower right", fontsize=10,
          title="Prospectivity Score", title_fontsize=10,
          framealpha=0.9, edgecolor="grey")
ax.set_xlabel("Easting (m)", fontsize=11)
ax.set_ylabel("Northing (m)", fontsize=11)
ax.set_title("Cordilleran REE Prospectivity (RF: Raster + Geology, No PCA)",
             fontsize=13, fontweight="bold")
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## 15. RF v3：原始 23 个化学元素 + raster + geology（无 PCA） 

In [ ]:
# RF v3：原始 23 个化学元素 + raster + geology（无 PCA）
# import rasterio, pyproj
# import matplotlib.colors as mcolors
# import matplotlib.patches as mpatches
# from scipy.interpolate import griddata as scipy_griddata
# from scipy.interpolate import griddata
# from rasterio.transform import Affine
# from rasterio.features import geometry_mask
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.impute import SimpleImputer
# from sklearn.metrics import roc_auc_score, classification_report
# from sklearn.model_selection import cross_val_score

# # ---- 1. 训练 ----
# cord_df_v3 = pd.read_csv("Cordilleran_integrated_features.csv")
# y_v3 = cord_df_v3["label"].values
# elem_cols_v3 = [c for c in ELEMENT_COLS if c in cord_df_v3.columns]
# skip_v3 = set(["lat", "lon", "label", "src", "geo_row", "t_name", "struct_proximity"])
# extra_v3 = [c for c in cord_df_v3.columns
#             if c not in skip_v3 and c not in elem_cols_v3
#             and cord_df_v3[c].dtype in ["float64", "int64", "float32", "int32"]]
# feat_v3 = elem_cols_v3 + extra_v3
# print(f"特征: {len(elem_cols_v3)} 化学元素 + {len(extra_v3)} 空间 = {len(feat_v3)} 个")
# imp_v3 = SimpleImputer(strategy="median")
# X_v3 = imp_v3.fit_transform(cord_df_v3[feat_v3].astype(float).values)
# rf_v3 = RandomForestClassifier(n_estimators=200, max_depth=15, min_samples_split=5,
#                                 min_samples_leaf=2, random_state=RANDOM_SEED,
#                                 n_jobs=-1, class_weight="balanced")
# rf_v3.fit(X_v3, y_v3)
# y_proba_v3 = rf_v3.predict_proba(X_v3)[:, 1]
# cv_auc_v3 = cross_val_score(rf_v3, X_v3, y_v3, cv=5, scoring="roc_auc", n_jobs=-1)
# print(f"Train AUC={roc_auc_score(y_v3, y_proba_v3):.3f}  CV AUC={cv_auc_v3.mean():.3f}±{cv_auc_v3.std():.3f}")
# feat_imp_v3 = pd.Series(rf_v3.feature_importances_, index=feat_v3).sort_values(ascending=False)
# print(feat_imp_v3.head(10).round(4))

# # ---- 2. 预测网格（0.1°，化学用 griddata 插值）----
# boundary_gdf = gpd.read_file("../dataset/Cordilleran.geojson").to_crs("EPSG:4326")
# bounds = boundary_gdf.total_bounds
# grid_spacing = 0.1
# xs = np.arange(bounds[0], bounds[2]+grid_spacing, grid_spacing)
# ys = np.arange(bounds[1], bounds[3]+grid_spacing, grid_spacing)
# xx, yy = np.meshgrid(xs, ys)
# grid_v3 = gpd.GeoDataFrame({"grid_id": range(len(xx.ravel()))},
#     geometry=gpd.points_from_xy(xx.ravel(), yy.ravel()), crs="EPSG:4326")
# grid_v3 = gpd.sjoin(grid_v3, boundary_gdf, how="inner", predicate="within")
# grid_v3 = grid_v3[["grid_id","geometry"]].drop_duplicates(subset=["grid_id"]).reset_index(drop=True)
# g_lons = grid_v3.geometry.x.values
# g_lats = grid_v3.geometry.y.values
# query_pts = np.column_stack([g_lons, g_lats])
# print(f"预测网格: {len(grid_v3)} 个格子（{grid_spacing}°）")
# grid_feats_v3 = pd.DataFrame(index=range(len(grid_v3)))

# # 化学元素：griddata 插值
# geo_df = pd.read_csv("../dataset/Geochemistry_Cordilleran_Clip.csv", low_memory=False).dropna(subset=["latitude","longitude"])
# geo_pts = np.column_stack([geo_df["longitude"].values, geo_df["latitude"].values])
# for elem in elem_cols_v3:
#     vals = geo_df[elem].values.astype(float)
#     mask = ~np.isnan(vals)
#     grid_feats_v3[elem] = scipy_griddata(geo_pts[mask], vals[mask], query_pts, method="linear") if mask.sum()>3 else np.nan
# print(f"化学插值完成，覆盖率: {grid_feats_v3[elem_cols_v3[0]].notna().mean()*100:.0f}%")

# # 栅格采样
# for feat_name, raster_path in REGIONS["Cordilleran"]["raster"].items():
#     with rasterio.open(raster_path) as src:
#         if src.crs and src.crs.to_string() != "EPSG:4326":
#             tr = pyproj.Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
#             xs_r, ys_r = tr.transform(g_lons, g_lats)
#             coords = list(zip(xs_r, ys_r))
#         else:
#             coords = list(zip(g_lons, g_lats))
#         vals = np.array([v[0] for v in src.sample(coords)], dtype=float)
#         if src.nodata is not None: vals[vals==src.nodata] = np.nan
#     grid_feats_v3[feat_name] = vals

# # struct_dist_m
# if "struct_dist_m" in extra_v3:
#     struct = gpd.read_file("../dataset/Cordilleran_structure.geojson").to_crs(PROJECTED_CRS)
#     su = struct.geometry.union_all() if hasattr(struct.geometry,"union_all") else struct.geometry.unary_union
#     grid_feats_v3["struct_dist_m"] = grid_v3.to_crs(PROJECTED_CRS).geometry.apply(lambda x: x.distance(su)).values

# # 地体
# terrane_feats_v3 = [c for c in ["geo_age","appearance"] if c in extra_v3]
# if terrane_feats_v3:
#     terr = gpd.read_file("../dataset/Cordilleran_Terrane.geojson").to_crs("EPSG:4326")
#     j = gpd.sjoin(grid_v3, terr[terrane_feats_v3+["geometry"]], how="left", predicate="within")
#     j = j[~j.index.duplicated(keep="first")]
#     for col in terrane_feats_v3:
#         vals_t = j[col].values.astype(float)
#         vals_t[vals_t<=-900] = np.nan
#         grid_feats_v3[col] = vals_t

# # ---- 3. 预测 ----
# X_grid_v3 = imp_v3.transform(grid_feats_v3.reindex(columns=feat_v3).astype(float).values)
# proba_v3 = rf_v3.predict_proba(X_grid_v3)[:,1]
# grid_v3["prospectivity"] = proba_v3
# print(f"Prospectivity: mean={proba_v3.mean():.3f}  max={proba_v3.max():.3f}")

# # ---- 4. 画图（裁剪到边界）----
# breaks = [0.0, round(float(np.percentile(proba_v3,25)),3),
#           round(float(np.percentile(proba_v3,50)),3),
#           round(float(np.percentile(proba_v3,75)),3), 1.0]
# colors_cls = ["#1a6faf","#42c4e8","#f5e21a","#d73027"]
# cmap_cls = mcolors.ListedColormap(colors_cls)
# norm_cls = mcolors.BoundaryNorm(breaks, len(colors_cls))

# fine_proj = grid_v3.to_crs(PROJECTED_CRS)
# xs_p, ys_p = fine_proj.geometry.x.values, fine_proj.geometry.y.values
# x_min, y_min, x_max, y_max = xs_p.min(), ys_p.min(), xs_p.max(), ys_p.max()
# res = int(grid_spacing * 111000)
# xi = np.arange(x_min, x_max, res); yi = np.arange(y_min, y_max, res)
# Xi, Yi = np.meshgrid(xi, yi)
# Z = griddata(np.column_stack([xs_p, ys_p]), proba_v3, (Xi, Yi), method="linear")

# # 裁剪到研究区边界
# _bnd = boundary_gdf.to_crs(PROJECTED_CRS)
# _transform = Affine.translation(x_min, y_max) * Affine.scale(res, -res)
# _mask = geometry_mask(_bnd.geometry, out_shape=Z.shape, transform=_transform, invert=True)
# Z = np.where(_mask, Z, np.nan)

# fig, ax = plt.subplots(figsize=(14, 11))
# ax.imshow(np.ma.masked_invalid(Z), extent=[x_min, x_max, y_min, y_max],
#           origin="upper", cmap=cmap_cls, norm=norm_cls, aspect="auto", alpha=0.9)
# boundary_gdf.to_crs(PROJECTED_CRS).boundary.plot(ax=ax, edgecolor="black", linewidth=1.5)
# dep = pd.read_csv("../dataset/CarbonatitehostREE_Cordilleran_Clip.csv", low_memory=False)
# gpd.GeoDataFrame(dep, geometry=gpd.points_from_xy(dep["Longitude"],dep["Latitude"]),
#                  crs="EPSG:4326").to_crs(PROJECTED_CRS).plot(
#     ax=ax, color="red", markersize=80, marker="*", alpha=0.45, zorder=5)
# legend_patches = [mpatches.Patch(facecolor=colors_cls[i], edgecolor="grey", linewidth=0.5,
#                                   label=f"{breaks[i]:.3f} - {breaks[i+1]:.3f}")
#                   for i in range(len(colors_cls))]
# legend_patches.append(plt.Line2D([0],[0],marker="*",color="w",markerfacecolor="red",
#                                    markersize=12,alpha=0.7,label="Known REE Occurrences"))
# ax.legend(handles=legend_patches, loc="lower right", fontsize=10,
#           title="Prospectivity Score", title_fontsize=10, framealpha=0.9, edgecolor="grey")
# ax.set_xlabel("Easting (m)", fontsize=11); ax.set_ylabel("Northing (m)", fontsize=11)
# ax.set_title("Cordilleran REE Prospectivity (RF v3: Raw 23 Elements + Raster + Geology)",
#              fontsize=12, fontweight="bold")
# ax.grid(True, alpha=0.3)
# plt.tight_layout(); plt.show()
# grid_v3.to_file("Cordilleran_prospectivity_v3_raw_chem.geojson", driver="GeoJSON")
# print("Saved: Cordilleran_prospectivity_v3_raw_chem.geojson")
# 

## 16. DevNet 半监督深度学习成矿预测模型

**DevNet（Deviation Network）** 是一种半监督异常检测框架：
- **已知矿床点** = 异常（高偏差分数）
- **背景采样点** = 正常（偏差分数接近参考分布 N(0,1)）
- 偏差损失（Deviation Loss）驱使矿床点的神经网络输出远离随机参考分布，背景点保持靠近
- 预测时对未知区域打分，高分 = 高成矿潜力

**网络结构**：Input → Dense(64, ReLU) → Dropout(0.3) → Dense(32, ReLU) → Dense(1, linear)

**特征集**：与 RF v1 完全相同（PC1-5 + raster + geology）

In [ ]:
# # DevNet 半监督深度学习 — 训练与评估
# import torch
# import torch.nn as nn
# import torch.optim as optim
# import numpy as np
# import matplotlib.pyplot as plt
# from sklearn.preprocessing import MinMaxScaler
# from sklearn.metrics import roc_auc_score
# from sklearn.model_selection import StratifiedKFold

# # ---- 0. 全局设置 ----
# DN_SEED = 42
# torch.manual_seed(DN_SEED)
# np.random.seed(DN_SEED)
# DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"设备: {DEVICE}")

# # ---- 1. 特征矩阵：PC1-5 + struct_dist_m + 地质 + 栅格（去掉 struct_proximity）----
# from sklearn.impute import SimpleImputer
# # extra_cols 来自 cell 19，去掉归一化不一致的 struct_proximity，保留 struct_dist_m
# dn1_extra_cols = [c for c in extra_cols if c != "struct_proximity"]
# dn1_extra_imp  = SimpleImputer(strategy="median")
# X_dn1_extra    = dn1_extra_imp.fit_transform(cord_df[dn1_extra_cols].astype(float).values)
# X_dn1_raw      = np.hstack([X_pca, X_dn1_extra]).astype(np.float32)
# dn1_feat_names = [f"PC{i+1}" for i in range(geo_pca.n_components_)] + dn1_extra_cols

# y_dn    = y_train.astype(np.float32)
# dn_scaler = MinMaxScaler()
# X_dn    = dn_scaler.fit_transform(X_dn1_raw).astype(np.float32)
# print(f"特征维度: {X_dn.shape}  正样本: {int(y_dn.sum())}  负样本: {int((1-y_dn).sum())}")
# print(f"特征列 ({len(dn1_feat_names)}): {dn1_feat_names}")

# # ---- 2. 网络定义 ----
# class DevNet(nn.Module):
#     def __init__(self, input_dim):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Linear(input_dim, 64), nn.ReLU(), nn.Dropout(0.3),
#             nn.Linear(64, 32),        nn.ReLU(), nn.Dropout(0.2),
#             nn.Linear(32, 1)
#         )
#     def forward(self, x):
#         return self.net(x).squeeze(-1)

# def deviation_loss(scores, labels, margin=5.0):
#     ref      = torch.randn(5000, device=scores.device)
#     ref_mean = ref.mean()
#     ref_std  = ref.std() + 1e-8
#     dev      = (scores - ref_mean) / ref_std
#     inlier   = torch.abs(dev)
#     outlier  = torch.clamp(margin - dev, min=0.0)
#     return ((1 - labels) * inlier + labels * outlier).mean()

# def make_batch(X_sc, pos_idx, neg_idx, batch_size, rng_st):
#     half  = batch_size // 2
#     idx   = np.concatenate([rng_st.choice(pos_idx, half, replace=True),
#                              rng_st.choice(neg_idx, half, replace=True)])
#     y_b   = np.array([1]*half + [0]*half, dtype=np.float32)
#     return torch.tensor(X_sc[idx]), torch.tensor(y_b)

# # ---- 3. 训练主模型 ----
# EPOCHS       = 900
# STEPS_PER_EP = 10
# BATCH_SIZE   = 128

# pos_idx  = np.where(y_dn == 1)[0]
# neg_idx  = np.where(y_dn == 0)[0]
# rng_main = np.random.RandomState(DN_SEED)

# model_dn  = DevNet(X_dn.shape[1]).to(DEVICE)
# optimizer = optim.Adam(model_dn.parameters(), lr=0.005, weight_decay=1e-4)
# scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# losses = []
# model_dn.train()
# for epoch in range(EPOCHS):
#     ep_loss = 0.0
#     for _ in range(STEPS_PER_EP):
#         Xb, yb = make_batch(X_dn, pos_idx, neg_idx, BATCH_SIZE, rng_main)
#         Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
#         optimizer.zero_grad()
#         loss = deviation_loss(model_dn(Xb), yb)
#         loss.backward()
#         optimizer.step()
#         ep_loss += loss.item()
#     scheduler.step()
#     losses.append(ep_loss / STEPS_PER_EP)
#     if (epoch + 1) % 60 == 0:
#         print(f"  Epoch {epoch+1:3d}/{EPOCHS}  loss={losses[-1]:.4f}")

# # ---- 4. 训练集 AUC ----
# model_dn.eval()
# with torch.no_grad():
#     all_scores = model_dn(torch.tensor(X_dn).to(DEVICE)).cpu().numpy()

# s_min, s_max = all_scores.min(), all_scores.max()
# train_proba  = (all_scores - s_min) / (s_max - s_min + 1e-8)
# train_auc    = roc_auc_score(y_dn, train_proba)
# print(f"\nTrain AUC = {train_auc:.4f}")

# # ---- 5. 5-fold 交叉验证 ----
# cv_aucs = []
# skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=DN_SEED)
# for fold, (tr_idx, val_idx) in enumerate(skf.split(X_dn, y_dn)):
#     torch.manual_seed(DN_SEED + fold)
#     rng_cv  = np.random.RandomState(DN_SEED + fold)
#     p_tr    = tr_idx[y_dn[tr_idx].astype(int) == 1]
#     n_tr    = tr_idx[y_dn[tr_idx].astype(int) == 0]
#     m       = DevNet(X_dn.shape[1]).to(DEVICE)
#     opt_cv  = optim.Adam(m.parameters(), lr=0.005, weight_decay=1e-4)
#     m.train()
#     for ep in range(150):
#         for _ in range(5):
#             Xb, yb = make_batch(X_dn, p_tr, n_tr, BATCH_SIZE, rng_cv)
#             Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
#             opt_cv.zero_grad()
#             deviation_loss(m(Xb), yb).backward()
#             opt_cv.step()
#     m.eval()
#     with torch.no_grad():
#         val_s = m(torch.tensor(X_dn[val_idx]).to(DEVICE)).cpu().numpy()
#     val_p = (val_s - val_s.min()) / (val_s.max() - val_s.min() + 1e-8)
#     cv_aucs.append(roc_auc_score(y_dn[val_idx], val_p))
#     print(f"  Fold {fold+1}  AUC={cv_aucs[-1]:.4f}")

# print(f"\nCV AUC: {np.mean(cv_aucs):.3f} ± {np.std(cv_aucs):.3f}")

# # ---- 6. 训练曲线 ----
# fig, ax = plt.subplots(figsize=(8, 3))
# ax.plot(losses, lw=1.5, color="#1a6faf")
# ax.set_xlabel("Epoch"); ax.set_ylabel("Deviation Loss")
# ax.set_title(f"DevNet 训练曲线  (Train AUC={train_auc:.3f}, CV AUC={np.mean(cv_aucs):.3f}±{np.std(cv_aucs):.3f})")
# ax.grid(True, alpha=0.3)
# plt.tight_layout(); plt.show()

# torch.save({"model_state": model_dn.state_dict(), "dn_scaler": dn_scaler,
#             "s_min": s_min, "s_max": s_max, "input_dim": X_dn.shape[1]}, "devnet_model.pt")
# print("模型已保存: devnet_model.pt")


In [ ]:
# # DevNet 成矿预测图 — 与 RF v1 相同的特征构建流程
# import torch
# import numpy as np
# import matplotlib.pyplot as plt
# import matplotlib.colors as mcolors
# import matplotlib.patches as mpatches
# import geopandas as gpd
# import pandas as pd
# from scipy.interpolate import griddata
# from scipy.spatial import cKDTree
# from rasterio.transform import Affine
# from rasterio.features import geometry_mask

# # ---- 1. 构建网格特征（与 cell 21 完全相同）----
# pc_cols    = [f"PC{i+1}" for i in range(geo_pca.n_components_)]
# pc_medians = np.median(X_pca, axis=0)
# X_pc_grid  = grid_features[pc_cols].values.copy()

# valid_mask = ~np.isnan(X_pc_grid[:, 0])
# if valid_mask.sum() > 3:
#     grid_proj = grid_gdf.to_crs(PROJECTED_CRS)
#     xs_all    = grid_proj.geometry.x.values
#     ys_all    = grid_proj.geometry.y.values
#     tree      = cKDTree(np.column_stack([xs_all[valid_mask], ys_all[valid_mask]]))
#     for i, col in enumerate(pc_cols):
#         nan_mask = np.isnan(X_pc_grid[:, i])
#         if nan_mask.sum() > 0:
#             nan_idx   = np.where(nan_mask)[0]
#             nan_pts   = np.column_stack([xs_all[nan_idx], ys_all[nan_idx]])
#             k         = min(5, valid_mask.sum())
#             dists, nb_idx = tree.query(nan_pts, k=k)
#             for j, (d, nidx) in enumerate(zip(dists, nb_idx)):
#                 w  = 1.0 / (d + 1e-8)
#                 vi = np.where(valid_mask)[0][nidx]
#                 X_pc_grid[nan_idx[j], i] = (w * X_pc_grid[vi, i]).sum() / w.sum()
# else:
#     for i in range(len(pc_cols)):
#         nan_mask = np.isnan(X_pc_grid[:, i])
#         X_pc_grid[nan_mask, i] = pc_medians[i]

# # 若 grid_features 还没有 struct_dist_m，实时计算（与训练集单位一致：米）
# if "struct_dist_m" not in grid_features.columns:
#     _struct = gpd.read_file("../dataset/Cordilleran_structure.geojson").to_crs(PROJECTED_CRS)
#     _su = _struct.geometry.union_all() if hasattr(_struct.geometry,"union_all") else _struct.geometry.unary_union
#     _gp  = grid_gdf.to_crs(PROJECTED_CRS)
#     print("计算预测网格 struct_dist_m（约1-2分钟）...")
#     grid_features["struct_dist_m"] = np.array([pt.distance(_su) for pt in _gp.geometry])
#     print(f"struct_dist_m: {grid_features['struct_dist_m'].min():.0f}–{grid_features['struct_dist_m'].max():.0f} m")

# X_extra_grid = dn1_extra_imp.transform(
#     grid_features.reindex(columns=dn1_extra_cols).astype(float).values)
# X_grid_full  = np.hstack([X_pc_grid, X_extra_grid]).astype(np.float32)

# # ---- 2. 归一化 + DevNet 推理 ----
# X_grid_sc = dn_scaler.transform(X_grid_full).astype(np.float32)
# model_dn.eval()
# with torch.no_grad():
#     grid_scores = model_dn(torch.tensor(X_grid_sc).to(DEVICE)).cpu().numpy()

# # 用网格自身的分数范围归一化（研究区内相对排名）
# g_min, g_max = grid_scores.min(), grid_scores.max()
# proba_dn = (grid_scores - g_min) / (g_max - g_min + 1e-8)
# print(f"grid_scores: min={g_min:.4f}  max={g_max:.4f}  std={grid_scores.std():.4f}")
# print(f"DevNet Prospectivity: mean={proba_dn.mean():.4f}  max={proba_dn.max():.3f}")

# # ---- 3. 投影 + 插值 + 边界裁剪 ----
# fine_proj = grid_gdf.to_crs(PROJECTED_CRS)
# xs_p, ys_p = fine_proj.geometry.x.values, fine_proj.geometry.y.values
# x_min_dn, y_min_dn = xs_p.min(), ys_p.min()
# x_max_dn, y_max_dn = xs_p.max(), ys_p.max()

# res_dn = 5000
# Xi_dn, Yi_dn = np.meshgrid(np.arange(x_min_dn, x_max_dn, res_dn),
#                              np.arange(y_min_dn, y_max_dn, res_dn))
# Z_dn = griddata(np.column_stack([xs_p, ys_p]), proba_dn, (Xi_dn, Yi_dn), method="linear")

# bnd_gdf = gpd.read_file("../dataset/Cordilleran.geojson").to_crs(PROJECTED_CRS)
# _tfm    = Affine.translation(x_min_dn, y_max_dn) * Affine.scale(res_dn, -res_dn)
# _mask   = geometry_mask(bnd_gdf.geometry, out_shape=Z_dn.shape, transform=_tfm, invert=True)
# Z_dn    = np.where(_mask, Z_dn, np.nan)

# # ---- 4. 绘图 ----
# valid  = proba_dn[~np.isnan(proba_dn)]

# # --- 图 A：上尾百分位断点（P90/P95/P99）---
# # 大部分背景（蓝）只有前 10% 才逐步变色
# bks_top = [0.0,
#            float(np.percentile(valid, 90)),
#            float(np.percentile(valid, 95)),
#            float(np.percentile(valid, 99)), 1.0]
# print(f"上尾断点: {[round(b,4) for b in bks_top]}")
# colors_cls  = ["#1a6faf","#42c4e8","#f5e21a","#d73027"]
# cmap_top    = mcolors.ListedColormap(colors_cls)
# norm_top    = mcolors.BoundaryNorm(bks_top, len(colors_cls))

# fig, axes = plt.subplots(1, 2, figsize=(22, 11))

# # 左图：上尾百分位（最有地质意义）
# ax = axes[0]
# ax.imshow(np.ma.masked_invalid(Z_dn), extent=[x_min_dn, x_max_dn, y_min_dn, y_max_dn],
#           origin="upper", cmap=cmap_top, norm=norm_top, aspect="auto", alpha=0.9)
# bnd_gdf.boundary.plot(ax=ax, edgecolor="black", linewidth=1.5)
# dep = pd.read_csv("../dataset/CarbonatitehostREE_Cordilleran_Clip.csv", low_memory=False)
# dep_gdf = gpd.GeoDataFrame(dep, geometry=gpd.points_from_xy(dep["Longitude"], dep["Latitude"]),
#                             crs="EPSG:4326").to_crs(PROJECTED_CRS)
# dep_gdf.plot(ax=ax, color="red", markersize=80, marker="*", alpha=0.45, zorder=5)
# patches = [mpatches.Patch(facecolor=colors_cls[i], edgecolor="grey", linewidth=0.5,
#                            label=f"P{[0,90,95,99,100][i]}–P{[0,90,95,99,100][i+1]} ({bks_top[i]:.4f}–{bks_top[i+1]:.4f})")
#            for i in range(4)]
# patches.append(plt.Line2D([0],[0], marker="*", color="w", markerfacecolor="red",
#                            markersize=12, alpha=0.7, label="Known REE"))
# ax.legend(handles=patches, loc="lower right", fontsize=9,
#           title="DevNet Score (上尾分位)", title_fontsize=9, framealpha=0.9)
# ax.set_title("DevNet 成矿潜力（上尾 P90/P95/P99 分类）", fontsize=12, fontweight="bold")
# ax.set_xlabel("Easting (m)"); ax.set_ylabel("Northing (m)")
# ax.grid(True, alpha=0.3)

# # 右图：平方根变换（全区显示梯度）
# Z_sqrt = np.sqrt(np.where(Z_dn >= 0, Z_dn, 0))
# bks_sqrt = [0.0,
#             float(np.percentile(np.sqrt(valid), 25)),
#             float(np.percentile(np.sqrt(valid), 50)),
#             float(np.percentile(np.sqrt(valid), 75)), 1.0]
# print(f"sqrt 四分位断点: {[round(b,3) for b in bks_sqrt]}")
# cmap_sqrt = mcolors.ListedColormap(colors_cls)
# norm_sqrt = mcolors.BoundaryNorm(bks_sqrt, len(colors_cls))

# ax2 = axes[1]
# ax2.imshow(np.ma.masked_invalid(Z_sqrt), extent=[x_min_dn, x_max_dn, y_min_dn, y_max_dn],
#            origin="upper", cmap=cmap_sqrt, norm=norm_sqrt, aspect="auto", alpha=0.9)
# bnd_gdf.boundary.plot(ax=ax2, edgecolor="black", linewidth=1.5)
# dep_gdf.plot(ax=ax2, color="red", markersize=80, marker="*", alpha=0.45, zorder=5)
# patches2 = [mpatches.Patch(facecolor=colors_cls[i], edgecolor="grey", linewidth=0.5,
#                             label=f"{bks_sqrt[i]:.3f}–{bks_sqrt[i+1]:.3f}")
#             for i in range(4)]
# patches2.append(plt.Line2D([0],[0], marker="*", color="w", markerfacecolor="red",
#                             markersize=12, alpha=0.7, label="Known REE"))
# ax2.legend(handles=patches2, loc="lower right", fontsize=9,
#            title="√score 四分位", title_fontsize=9, framealpha=0.9)
# ax2.set_title("DevNet 成矿潜力（√变换后四分位，显示全区梯度）", fontsize=12, fontweight="bold")
# ax2.set_xlabel("Easting (m)")
# ax2.grid(True, alpha=0.3)

# plt.suptitle("Cordilleran REE Prospectivity — DevNet Semi-Supervised", fontsize=13, fontweight="bold")
# plt.tight_layout(); plt.show()

# grid_out = grid_gdf.copy()
# grid_out["prospectivity_dn"] = proba_dn
# grid_out.to_file("Cordilleran_prospectivity_devnet.geojson", driver="GeoJSON")
# print("Saved: Cordilleran_prospectivity_devnet.geojson")


## # DevNet v2：Raster + Geology + 余弦相似度最优负样本选择 不用 PCA/地化 → 负样本可从整个网格取，用余弦距离选最不像矿床的点

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt, matplotlib.colors as mcolors, matplotlib.patches as mpatches
from scipy.interpolate import griddata
from scipy.spatial import cKDTree
from pyproj import Transformer
from rasterio.transform import Affine
from rasterio.features import geometry_mask
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

DN2_SEED = 42
torch.manual_seed(DN2_SEED); np.random.seed(DN2_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"设备: {DEVICE}")

# ---- 1. 特征集：Raster + Geology，无 PCA ----
raster_cols = list(REGIONS["Cordilleran"]["raster"].keys())
geo_cols    = [c for c in ["geo_age", "appearance"] if c in grid_features.columns]

# 若 grid_features 还没有 struct_dist_m，实时计算
if "struct_dist_m" not in grid_features.columns:
    _struct = gpd.read_file("../dataset/Cordilleran_structure.geojson").to_crs(PROJECTED_CRS)
    _su = _struct.geometry.union_all() if hasattr(_struct.geometry,"union_all") else _struct.geometry.unary_union
    _gp  = grid_gdf.to_crs(PROJECTED_CRS)
    print("计算预测网格 struct_dist_m（约1-2分钟）...")
    grid_features["struct_dist_m"] = np.array([pt.distance(_su) for pt in _gp.geometry])
    print(f"struct_dist_m: {grid_features['struct_dist_m'].min():.0f}–{grid_features['struct_dist_m'].max():.0f} m")

feat_dn2    = raster_cols + geo_cols + ["struct_dist_m"]
print(f"特征 ({len(feat_dn2)}): {feat_dn2}")

# ---- 2. 正样本：从训练 CSV 读取 ----
cord_df2 = pd.read_csv("Cordilleran_integrated_features.csv")
pos_df   = cord_df2[cord_df2["label"] == 1].reset_index(drop=True)
X_pos    = pos_df[feat_dn2].astype(float).values
pos_medians = np.nanmedian(X_pos, axis=0)
for j in range(X_pos.shape[1]):
    X_pos[np.isnan(X_pos[:, j]), j] = pos_medians[j]
print(f"正样本: {len(pos_df)} 个")

# ---- 3. 候选负样本：整个预测网格（已有 raster+geology 特征）----
cand_medians = np.nanmedian(grid_features[feat_dn2].astype(float).values, axis=0)
X_cand  = grid_features[feat_dn2].astype(float).values.copy()
for j in range(X_cand.shape[1]):
    X_cand[np.isnan(X_cand[:, j]), j] = cand_medians[j]
cand_lons = grid_gdf.geometry.x.values
cand_lats = grid_gdf.geometry.y.values
print(f"候选池: {len(X_cand)} 个网格点")

# ---- 4. 缓冲区过滤：剔除离矿床 50 km 内的格子 ----
tr_crs = Transformer.from_crs("EPSG:4326", PROJECTED_CRS, always_xy=True)
dep_x, dep_y   = tr_crs.transform(pos_df["lon"].values, pos_df["lat"].values)
cand_x, cand_y = tr_crs.transform(cand_lons, cand_lats)
dep_tree = cKDTree(np.column_stack([dep_x, dep_y]))
dists, _ = dep_tree.query(np.column_stack([cand_x, cand_y]), k=1)
BUFFER_M = 50_000  # 50 km
outside  = dists > BUFFER_M
X_cand_out   = X_cand[outside]
cand_idx_out = np.where(outside)[0]
print(f"缓冲区外候选点: {outside.sum()} 个")

# ---- 5. 余弦相似度选负样本：取与矿床最不相似的点 ----
def norm_rows(A):
    n = np.linalg.norm(A, axis=1, keepdims=True) + 1e-8
    return A / n

cos_mat   = norm_rows(X_cand_out) @ norm_rows(X_pos).T  # (n_cand, n_pos)
mean_cos  = cos_mat.mean(axis=1)                         # 每个候选点对所有正样本的均值
sorted_i  = np.argsort(mean_cos)                         # 升序：cos 越小越不像矿床

N_NEG     = min(1000, len(X_cand_out))
neg_sel   = sorted_i[:N_NEG]                              # 最不相似的 N_NEG 个
X_neg     = X_cand_out[neg_sel]
neg_grid_idx = cand_idx_out[neg_sel]

print(f"负样本: {N_NEG} 个  (mean cos={mean_cos[neg_sel].mean():.3f}，"
      f"随机抽取均值≈{mean_cos.mean():.3f})")

# cos 分布图
fig_cos, ax_cos = plt.subplots(figsize=(7, 3))
ax_cos.hist(mean_cos, bins=50, color="#1a6faf", alpha=0.7, label="全部候选")
ax_cos.axvline(mean_cos[neg_sel].max(), color="#d73027", lw=2,
               label=f"选取阈值 (N={N_NEG})")
ax_cos.set_xlabel("与矿床的平均余弦相似度"); ax_cos.set_ylabel("格子数量")
ax_cos.set_title("负样本选取：最低余弦相似度（最不像矿床）")
ax_cos.legend(); plt.tight_layout(); plt.show()

# ---- 6. 合并训练集 ----
X_dn2_raw = np.vstack([X_pos, X_neg])
y_dn2     = np.array([1]*len(X_pos) + [0]*N_NEG, dtype=np.float32)
dn2_scaler = MinMaxScaler()
X_dn2      = dn2_scaler.fit_transform(X_dn2_raw).astype(np.float32)
print(f"训练集: {X_dn2.shape}  正:{int(y_dn2.sum())}  负:{int((1-y_dn2).sum())}")

# ---- 7. DevNet 网络 & 损失 ----
class DevNet2(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 1))
    def forward(self, x): return self.net(x).squeeze(-1)

def dev_loss2(scores, labels, margin=5.0):
    ref = torch.randn(5000, device=scores.device)
    dev = (scores - ref.mean()) / (ref.std() + 1e-8)
    return ((1 - labels)*torch.abs(dev) + labels*torch.clamp(margin-dev, min=0)).mean()

def batch2(X, pi, ni, bs, rng):
    h = bs // 2
    idx = np.concatenate([rng.choice(pi, h, replace=True), rng.choice(ni, h, replace=True)])
    yb  = np.array([1]*h + [0]*h, dtype=np.float32)
    return torch.tensor(X[idx]), torch.tensor(yb)

# ---- 8. 训练主模型 ----
EPOCHS2=900; STEPS2=10; BATCH2=128
pi2 = np.where(y_dn2==1)[0]; ni2 = np.where(y_dn2==0)[0]
rng2 = np.random.RandomState(DN2_SEED)
model_dn2 = DevNet2(X_dn2.shape[1]).to(DEVICE)
opt2 = optim.Adam(model_dn2.parameters(), lr=0.005, weight_decay=1e-4)
sch2 = optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=EPOCHS2)
losses2 = []
model_dn2.train()
for ep in range(EPOCHS2):
    el = 0.0
    for _ in range(STEPS2):
        Xb, yb = batch2(X_dn2, pi2, ni2, BATCH2, rng2)
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        opt2.zero_grad(); dev_loss2(model_dn2(Xb), yb).backward(); opt2.step()
        el += dev_loss2(model_dn2(Xb), yb).item()
    sch2.step(); losses2.append(el/STEPS2)
    if (ep+1) % 60 == 0: print(f"  Epoch {ep+1}/{EPOCHS2}  loss={losses2[-1]:.4f}")

model_dn2.eval()
with torch.no_grad():
    ts = model_dn2(torch.tensor(X_dn2).to(DEVICE)).cpu().numpy()
tp = (ts-ts.min())/(ts.max()-ts.min()+1e-8)
print(f"\nTrain AUC: {roc_auc_score(y_dn2, tp):.4f}")

# 5-fold CV
cv2 = []
for fold,(tri,vli) in enumerate(StratifiedKFold(5,shuffle=True,random_state=DN2_SEED).split(X_dn2,y_dn2)):
    torch.manual_seed(DN2_SEED+fold); rng_cv=np.random.RandomState(DN2_SEED+fold)
    pt=tri[y_dn2[tri].astype(int)==1]; nt=tri[y_dn2[tri].astype(int)==0]
    m2=DevNet2(X_dn2.shape[1]).to(DEVICE); o2=optim.Adam(m2.parameters(),lr=0.005,weight_decay=1e-4)
    m2.train()
    for ep in range(150):
        for _ in range(5):
            Xb,yb=batch2(X_dn2,pt,nt,BATCH2,rng_cv); Xb,yb=Xb.to(DEVICE),yb.to(DEVICE)
            o2.zero_grad(); dev_loss2(m2(Xb),yb).backward(); o2.step()
    m2.eval()
    with torch.no_grad(): vs=m2(torch.tensor(X_dn2[vli]).to(DEVICE)).cpu().numpy()
    vp=(vs-vs.min())/(vs.max()-vs.min()+1e-8)
    cv2.append(roc_auc_score(y_dn2[vli], vp))
    print(f"  Fold {fold+1}  AUC={cv2[-1]:.4f}")
print(f"\nCV AUC: {np.mean(cv2):.3f} ± {np.std(cv2):.3f}")

# 训练曲线
fig_l, ax_l = plt.subplots(figsize=(8,3))
ax_l.plot(losses2, lw=1.5, color="#1a6faf")
ax_l.set_xlabel("Epoch"); ax_l.set_ylabel("Deviation Loss")
ax_l.set_title(f"DevNet v2 训练曲线  CV AUC={np.mean(cv2):.3f}±{np.std(cv2):.3f}")
ax_l.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

# ---- 9. 预测全网格 ----
X_grid_dn2 = grid_features[feat_dn2].astype(float).values.copy()
for j in range(X_grid_dn2.shape[1]):
    col_nan = np.isnan(X_grid_dn2[:, j])
    X_grid_dn2[col_nan, j] = cand_medians[j]      # 列中位数填充
X_grid_dn2 = dn2_scaler.transform(X_grid_dn2).astype(np.float32)

model_dn2.eval()
with torch.no_grad():
    gs2 = model_dn2(torch.tensor(X_grid_dn2).to(DEVICE)).cpu().numpy()

# 步骤A：min-max 归一化到 [0,1]（研究区内相对排名）
g2min, g2max = gs2.min(), gs2.max()
proba_dn2 = (gs2 - g2min) / (g2max - g2min + 1e-8)
print(f"raw scores  : min={g2min:.4f}  max={g2max:.4f}  std={gs2.std():.4f}")
print(f"proba_dn2   : mean={proba_dn2.mean():.4f}  max={proba_dn2.max():.4f}")

# 步骤B：幂变换（仅用于可视化，拉伸低值区使空间梯度可见）
POWER = 0.2          # 指数越小，拉伸越强；可在 0.15~0.3 范围调试
proba_viz = proba_dn2 ** POWER
print(f"proba_viz   : mean={proba_viz.mean():.3f}  (power={POWER}, 仅用于着色)")

# ---- 10. 成矿预测图 ----
fp2 = grid_gdf.to_crs(PROJECTED_CRS)
xs2, ys2 = fp2.geometry.x.values, fp2.geometry.y.values
xmn2,ymn2,xmx2,ymx2 = xs2.min(),ys2.min(),xs2.max(),ys2.max()
r2 = 5000
Xi2,Yi2 = np.meshgrid(np.arange(xmn2,xmx2,r2), np.arange(ymn2,ymx2,r2))

# 用幂变换后的分数做插值（颜色用）
Z2_viz = griddata(np.column_stack([xs2,ys2]), proba_viz, (Xi2,Yi2), method="linear")
bnd2 = gpd.read_file("../dataset/Cordilleran.geojson").to_crs(PROJECTED_CRS)
tfm2 = Affine.translation(xmn2,ymx2)*Affine.scale(r2,-r2)
msk2 = geometry_mask(bnd2.geometry, out_shape=Z2_viz.shape, transform=tfm2, invert=True)
Z2_viz = np.where(msk2, Z2_viz, np.nan)

# 四分位断点：基于 proba_viz（变换后均匀分布，P25/P50/P75 有意义）
valid_viz = proba_viz[~np.isnan(proba_viz)]
bks_viz = [0.0,
           float(np.percentile(valid_viz, 90)),
           float(np.percentile(valid_viz, 95)),
           float(np.percentile(valid_viz, 98)), 1.0]
# 转换回原始分数供图例显示
bks_raw = [round(b**( 1/POWER), 4) for b in bks_viz]
print(f"viz 断点 (变换后): {[round(b,3) for b in bks_viz]}")
print(f"对应原始分数    : {bks_raw}")

colors_cls = ["#1a6faf","#42c4e8","#f5e21a","#d73027"]
cmap2 = mcolors.ListedColormap(colors_cls)
norm2 = mcolors.BoundaryNorm(bks_viz, len(colors_cls))

fig, ax = plt.subplots(figsize=(14, 11))
ax.imshow(np.ma.masked_invalid(Z2_viz),
          extent=[xmn2,xmx2,ymn2,ymx2],
          origin="upper", cmap=cmap2, norm=norm2, aspect="auto", alpha=0.9)
bnd2.boundary.plot(ax=ax, edgecolor="black", linewidth=1.5)

dep = pd.read_csv("../dataset/CarbonatitehostREE_Cordilleran_Clip.csv", low_memory=False)
gpd.GeoDataFrame(dep, geometry=gpd.points_from_xy(dep["Longitude"],dep["Latitude"]),
                 crs="EPSG:4326").to_crs(PROJECTED_CRS).plot(
    ax=ax, color="red", markersize=80, marker="*", alpha=0.45, zorder=5)

patches2 = [mpatches.Patch(facecolor=colors_cls[i], edgecolor="grey", linewidth=0.5,
                            label=f"Q{i+1}  score≥{bks_raw[i]:.4f}")
            for i in range(4)]
patches2.append(plt.Line2D([0],[0],marker="*",color="w",markerfacecolor="red",
                            markersize=12,alpha=0.7,label="Known REE"))
ax.legend(handles=patches2, loc="lower right", fontsize=9,
          title=f"DevNet v2 (幂变换可视化, power={POWER})", title_fontsize=9, framealpha=0.9)
ax.set_xlabel("Easting (m)"); ax.set_ylabel("Northing (m)")
ax.set_title(f"Cordilleran REE 成矿潜力 — DevNet v2\n"
             f"(Raster+Geology, {N_NEG}负样本余弦最优, proba^{POWER} 拉伸可视化)",
             fontsize=12, fontweight="bold")
ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

# 保存原始归一化分数（不是变换后的）
gout2 = grid_gdf.copy()
gout2["prospectivity_dn2"] = proba_dn2
gout2.to_file("Cordilleran_prospectivity_devnet_v2.geojson", driver="GeoJSON")
print("Saved: Cordilleran_prospectivity_devnet_v2.geojson  (保存原始 proba_dn2，非变换值)")
